<a href="https://colab.research.google.com/github/iantextbar/previsao_curto_prazo_consumo_energia_eletrica_MAE5920/blob/refactor%2Fvalidacao-temporal/previsao_curto_prazo_consumo_energia_eletrica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

__Execução no Google Colab (opcional).__ A célula abaixo só faz algo quando o notebook roda no Colab: clona o repositório, entra na pasta do notebook (para que os caminhos relativos funcionem) e instala as dependências que faltam no ambiente do Colab. Localmente, ela não faz nada. Para a busca completa da GRU, use um ambiente com GPU.

In [1]:
import os
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/iantextbar/previsao_curto_prazo_consumo_energia_eletrica_MAE5920.git'
REPO_BRANCH = 'refactor/validacao-temporal'

if IN_COLAB:
    repo_dir = '/content/previsao_curto_prazo_consumo_energia_eletrica_MAE5920'
    if not os.path.exists(repo_dir):
        subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, repo_dir], check=True)
    os.chdir(f'{repo_dir}/src/notebooks')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'statsmodels>=0.15', 'plotly>=6'], check=True)

In [2]:
import random
import warnings
import joblib
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import torch

from copy import deepcopy
from datetime import datetime
from itertools import product
from pathlib import Path
from time import perf_counter
from numpy.lib.stride_tricks import sliding_window_view
from plotly.subplots import make_subplots
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tools.sm_exceptions import InterpolationWarning
from statsmodels.tsa.stattools import acf, kpss, pacf
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# __Previsão de curto prazo do consumo residencial de energia elétrica__
## __Universidade de São Paulo__

### *Disciplina: Deep Learning - MAE5920 - Prof. Dr. Hamed Yazdanpanah*
### *Ian Teixeira Barreiro (10717526) e Ornella Scardua Ferreira (ADD NUSP)*

_Fonte:_ https://archive.ics.uci.edu/dataset/374/appliances+energy+prediction

_Descrição dos Dados:_ The data set is at 10 min for about 4.5 months. The house temperature and humidity conditions were monitored with a ZigBee wireless sensor network. Each wireless node transmitted the temperature and humidity conditions around 3.3 min. Then, the wireless data was averaged for 10 minutes periods. The energy data was logged every 10 minutes with m-bus energy meters. Weather from the nearest airport weather station (Chievres Airport, Belgium) was downloaded from a public data set from Reliable Prognosis (rp5.ru), and merged together with the experimental data sets using the date and time column. Two random variables have been included in the data set for testing the regression models and to filter out non predictive attributes (parameters).

_Descrição da tarefa:_ Prever o consumo de energia dos eletrodomésticos com horizontes de 10 e 60 minutos.

__Modelos sugeridos:__

    - Baselines: regressão linear autorregressiva com variáveis exógenas.

    - Modelo profundo: rede GRU ou rede neural convolucional unidimensional (CNN 1D) de
pequeno porte.

__Avaliação:__

    - Erro absoluto médio (MAE) e raiz do erro quadrático médio (RMSE).

    - Erro absoluto médio escalonado (MASE), utilizando a previsão por persistência como referência.

    - Desempenho das previsões por hora do dia.

__Entregáveis finais:__

    - Um notebook ou projeto Python reproduzível.

    - Um dicionário de dados resumido e uma análise de possíveis vazamentos de informação
(data leakage).

    - Implementações de modelos de referência (baselines) e de aprendizagem profunda.

    - Uma tabela com os resultados obtidos em dados de treino, validação e teste.

    - Análises de calibração ou diagnósticos dos resíduos.

    - Um resumo do custo computacional.

    - Um relatório de aproximadamente 6 a 8 páginas.

    - Uma apresentação de 10 a 12 minutos.

    - Uma seção final intitulada “Quando o modelo de aprendizagem profunda falha?”.

## Data Loading

In [3]:
ROOT = Path(__name__).resolve().parent.parent
DATAPATH = ROOT / 'data'

original_df = pd.read_csv(f'{DATAPATH}/energydata_complete.csv')

## EDA

In [4]:
original_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19735 entries, 0 to 19734
Data columns (total 29 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         19735 non-null  object 
 1   Appliances   19735 non-null  int64  
 2   lights       19735 non-null  int64  
 3   T1           19735 non-null  float64
 4   RH_1         19735 non-null  float64
 5   T2           19735 non-null  float64
 6   RH_2         19735 non-null  float64
 7   T3           19735 non-null  float64
 8   RH_3         19735 non-null  float64
 9   T4           19735 non-null  float64
 10  RH_4         19735 non-null  float64
 11  T5           19735 non-null  float64
 12  RH_5         19735 non-null  float64
 13  T6           19735 non-null  float64
 14  RH_6         19735 non-null  float64
 15  T7           19735 non-null  float64
 16  RH_7         19735 non-null  float64
 17  T8           19735 non-null  float64
 18  RH_8         19735 non-null  float64
 19  T9  

In [5]:
original_df.head()

,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
0,2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,...,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
1,2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,...,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195
2,2016-01-11 17:20:00,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,...,17.000000,45.50,6.366667,733.7,92.0,6.333333,55.333333,5.1,28.642668,28.642668
3,2016-01-11 17:30:00,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,...,17.000000,45.40,6.250000,733.8,92.0,6.000000,51.500000,5.0,45.410389,45.410389
4,2016-01-11 17:40:00,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,...,17.000000,45.40,6.133333,733.9,92.0,5.666667,47.666667,4.9,10.084097,10.084097


Vemos que não há dados faltantes em nenhum das variáveis. Há uma variável de datas que está no formato datetime mas está como str. As demais variáveis são inteiras ou float. Segue uma descrição das variáveis.

__Variables:__

    - Appliances, energy use in Wh
    - lights, energy use of light fixtures in the house in Wh
    - T1, Temperature in kitchen area, in Celsius
    - RH_1, Humidity in kitchen area, in %
    - T2, Temperature in living room area, in Celsius
    - RH_2, Humidity in living room area, in %
    - T3, Temperature in laundry room area
    - RH_3, Humidity in laundry room area, in %
    - T4, Temperature in office room, in Celsius
    - RH_4, Humidity in office room, in %
    - T5, Temperature in bathroom, in Celsius
    - RH_5, Humidity in bathroom, in %
    - T6, Temperature outside the building (north side), in Celsius
    - RH_6, Humidity outside the building (north side), in %
    - T7, Temperature in ironing room , in Celsius
    - RH_7, Humidity in ironing room, in %
    - T8, Temperature in teenager room 2, in Celsius
    - RH_8, Humidity in teenager room 2, in %
    - T9, Temperature in parents room, in Celsius
    - RH_9, Humidity in parents room, in %
    - To, Temperature outside (from Chievres weather station), in Celsius
    - Pressure (from Chievres weather station), in mm Hg
    - RH_out, Humidity outside (from Chievres weather station), in %
    - Wind speed (from Chievres weather station), in m/s
    - Visibility (from Chievres weather station), in km
    - Tdewpoint (from Chievres weather station), Â°C
    - rv1, Random variable 1, nondimensional
    - rv2, Random variable 2, nondimensional

Ou seja, as variáveis indicam consumo de energia elétrica por luzes e eletrodomésticos, humidade e temperatura em cada cômodo, condições climáticas e duas variáveis aleatórias.

Vamos adianter um pequeno passo de preprocessamento para facilitar o EDA, tranformando as datas em datetime.

In [6]:
original_df['date'] = pd.to_datetime(original_df['date'], format='%Y-%m-%d %H:%M:%S')
original_df = original_df.set_index('date', drop=False)

print(original_df['date'].dtype)
print(f"Período: {original_df['date'].min()} até {original_df['date'].max()}")
print(f"Intervalos entre observações: {original_df['date'].diff().value_counts().to_dict()}")

datetime64[ns]
Período: 2016-01-11 17:00:00 até 2016-05-27 18:00:00
Intervalos entre observações: {Timedelta('0 days 00:10:00'): 19734}


Usaremos o mapeamento abaixo para dar nomes legíveis aos cômodos nos gráficos. Note que `T6`/`RH_6` são medidos do lado de fora do prédio (lado norte), portanto não entram como cômodos da casa.

In [7]:
ROOMS = {
    '1': 'Cozinha',
    '2': 'Sala de estar',
    '3': 'Lavanderia',
    '4': 'Escritório',
    '5': 'Banheiro',
    '7': 'Sala de passar',
    '8': 'Quarto adolescente',
    '9': 'Quarto dos pais',
}
TEMP_COLS = [f'T{i}' for i in ROOMS]
HUM_COLS = [f'RH_{i}' for i in ROOMS]

original_df[['Appliances', 'lights'] + TEMP_COLS + HUM_COLS + ['T6', 'RH_6', 'T_out', 'RH_out']].describe().T

,count,mean,std,min,25%,50%,75%,max
Appliances,19735.0,97.694958,102.524891,10.000000,50.000000,60.000000,100.000000,1080.000000
lights,19735.0,3.801875,7.935988,0.000000,0.000000,0.000000,0.000000,70.000000
T1,19735.0,21.686571,1.606066,16.790000,20.760000,21.600000,22.600000,26.260000
T2,19735.0,20.341219,2.192974,16.100000,18.790000,20.000000,21.500000,29.856667
T3,19735.0,22.267611,2.006111,17.200000,20.790000,22.100000,23.290000,29.236000
T4,19735.0,20.855335,2.042884,15.100000,19.530000,20.666667,22.100000,26.200000
T5,19735.0,19.592106,1.844623,15.330000,18.277500,19.390000,20.619643,25.795000
T7,19735.0,20.267106,2.109993,15.390000,18.700000,20.033333,21.600000,26.000000
T8,19735.0,22.029107,1.956162,16.306667,20.790000,22.100000,23.390000,27.230000
T9,19735.0,19.485828,2.014712,14.890000,18.000000,19.390000,20.600000,24.500000


Podemos ver que há uma diferença nas médias de temperaturas entre os cômodos. Comparando médias e medianas, a distribuição das temperaturas de todos os cômodos parece ser simétrica. Mas alguns cômodos apresentam desvio-padrão um pouco maior que outros. O mesmo ocorre para humidade.

Para eletrodoméstricos e luzes, vemos significativa assimetria. Também vemos, como esperado, que eletrodomésticos consomem mais energia elétrica que luzes, e com maior variância.

### Consumo de energia ao longo do tempo

A série original tem resolução de 10 minutos e é bastante ruidosa, então plotamos também a média diária para evidenciar a tendência ao longo dos ~4,5 meses (de janeiro a maio).

In [8]:
daily = original_df[['Appliances', 'lights']].resample('D').mean()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=('Eletrodomésticos (Wh)', 'Iluminação (Wh)'))
for row, col in enumerate(['Appliances', 'lights'], start=1):
    fig.add_trace(go.Scattergl(x=original_df.index, y=original_df[col], mode='lines', name=f'{col} (10 min)',
                               line=dict(width=0.5), opacity=0.4), row=row, col=1)
    fig.add_trace(go.Scatter(x=daily.index, y=daily[col], mode='lines', name=f'{col} (média diária)',
                             line=dict(width=2)), row=row, col=1)
fig.update_layout(height=650, title='Consumo de energia ao longo do tempo')
fig.show()

Como podemos ver, em cada dia no grafo de 10 em 10 minutos há bastante oscilação no consumo de energia elétrica, indicando que seria interessante observar o comporamento diário. Visualmente, é possível ver que há dias de maior consumo tanto de energia por eletrodomésticos quanto por luzes. Não é possível detectar nenhuma sazonalidade visualmente por enquanto.

Como a tarefa envolve avaliar o desempenho por hora do dia, vale olhar o perfil médio de consumo por hora e por dia da semana.

In [9]:
profile = original_df[['Appliances', 'lights']].assign(
    hora=original_df.index.hour,
    dia_semana=original_df.index.day_name(),
)

fig = px.line(profile.groupby('hora')[['Appliances', 'lights']].mean(), markers=True,
              title='Consumo médio por hora do dia', labels={'value': 'Wh', 'variable': ''})
fig.show()

weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heat = profile.pivot_table(index='dia_semana', columns='hora', values='Appliances', aggfunc='mean').loc[weekday_order]
fig = px.imshow(heat, aspect='auto', color_continuous_scale='Viridis',
                title='Consumo médio dos eletrodomésticos (Wh) por dia da semana e hora',
                labels={'x': 'Hora', 'y': '', 'color': 'Wh'})
fig.show()

Vemos que há baixo consumo de energia de manhã e à noite. Durante a manhã, o consumo vai aumentando. À partir das 16 horas, no final da tarde, o consumo vai aumentando, tendo seu pico às 18 horas. À partir desse horário o consumo vai baixando. Tal comportamento pode ser facilmente lido em termos comportamentais.

Vemos que às 18 horas, para todos os dias da semana há esse pico, provavelmente correspondendo ao horário de retorno do trabalho e início dos a fazeres domésticos. Pela tarde, na terça-feira, quarta-feira, quinta-feira e domingo há menos consumo de energia por eletrodomésticos, potencialmente por serem dias que as pessoas não estão em casa. Possivelmente, segunda-feira e sexta são dias de home-office, pelo consumo alto pela tarde.

In [10]:
fig = px.histogram(original_df, x='Appliances', nbins=100, title='Distribuição do consumo dos eletrodomésticos')
fig.show()

Vemos que há dias de altíssimo consumo por appliances, mas esses são outliers.

### Temperatura nos cômodos

Para não sobrecarregar o gráfico com 8 séries de ~20 mil pontos, usamos a média horária. O boxplot ao lado permite comparar a distribuição de temperatura de cada cômodo.

In [11]:
def plot_rooms(cols, prefix, unit, title):
    hourly = (original_df[cols].resample('h').mean()
              .rename(columns=lambda c: ROOMS[c.removeprefix(prefix)]))
    fig = px.line(hourly, title=f'{title} por cômodo (média horária)',
                  labels={'value': unit, 'date': '', 'variable': 'Cômodo'})
    fig.show()

    long = (original_df[cols].rename(columns=lambda c: ROOMS[c.removeprefix(prefix)])
            .melt(var_name='Cômodo', value_name=unit))
    fig = px.box(long, x='Cômodo', y=unit, color='Cômodo', title=f'Distribuição de {title.lower()} por cômodo')
    fig.update_layout(showlegend=False)
    fig.show()


plot_rooms(TEMP_COLS, 'T', '°C', 'Temperatura')

Percebe-se que na média, os cômodos exibem comportamento semelhante em termos de temperatura, como de se esperar pela influência da temperatura externa. No entanto, há cômodos com médias diferentes, e variâncias diferentes, como já haviamos percebido. É interessante perceber a grande quantidade de pontos extremos positivos na sala de estar e na lavanderia, indicando momentos de temperatura atípica nesses cômodos.

### Umidade nos cômodos

In [12]:
plot_rooms(HUM_COLS, 'RH_', '%', 'Umidade')

Banheiro óbviamente bem mais úmido. A sala de passar é mais seca que a média. Demais cômodos possuem distribuição muito parecida. Cozinha tem momentos de alta umidade, como de se esperar dependendo do que se cozinha. Sala de estar tem momentos de baixa umidade. Potencialmente pelo uso de ar condicionado ou ventiladores?

### Condições externas

Comparamos a temperatura e umidade da estação meteorológica de Chièvres (`T_out`, `RH_out`) com as do sensor instalado do lado de fora do prédio (`T6`, `RH_6`).

In [13]:
hourly_out = original_df[['T_out', 'T6', 'RH_out', 'RH_6']].resample('h').mean()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=('Temperatura externa (°C)', 'Umidade externa (%)'))
fig.add_trace(go.Scatter(x=hourly_out.index, y=hourly_out['T_out'], name='T_out (estação)'), row=1, col=1)
fig.add_trace(go.Scatter(x=hourly_out.index, y=hourly_out['T6'], name='T6 (lado norte)'), row=1, col=1)
fig.add_trace(go.Scatter(x=hourly_out.index, y=hourly_out['RH_out'], name='RH_out (estação)'), row=2, col=1)
fig.add_trace(go.Scatter(x=hourly_out.index, y=hourly_out['RH_6'], name='RH_6 (lado norte)'), row=2, col=1)
fig.update_layout(height=650, title='Temperatura e umidade externas (média horária)')
fig.show()

Vemos que a temperatura é praticamente idêntica. No entanto, à partir de um certo momento, a umidade na estação é consistentemente maior que a do lado de fora da casa.

### Diferença entre temperatura interna e externa

A temperatura média da casa é a média das temperaturas dos 8 cômodos (soma das temperaturas dividida pelo número de cômodos). A diferença `T_casa - T_out` indica o quanto a casa está mais quente que o ambiente externo.

In [14]:
temp_diff = pd.DataFrame({
    'T_casa': original_df[TEMP_COLS].mean(axis=1),
    'T_out': original_df['T_out'],
})
temp_diff['T_casa - T_out'] = temp_diff['T_casa'] - temp_diff['T_out']
hourly_diff = temp_diff.resample('h').mean()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=('Temperatura média da casa vs. externa (°C)', 'Diferença T_casa - T_out (°C)'))
fig.add_trace(go.Scatter(x=hourly_diff.index, y=hourly_diff['T_casa'], name='T_casa'), row=1, col=1)
fig.add_trace(go.Scatter(x=hourly_diff.index, y=hourly_diff['T_out'], name='T_out'), row=1, col=1)
fig.add_trace(go.Scatter(x=hourly_diff.index, y=hourly_diff['T_casa - T_out'], name='T_casa - T_out',
                         fill='tozeroy'), row=2, col=1)
fig.update_layout(height=650, title='Temperatura interna vs. externa (média horária)')
fig.show()

Aqui devemos desconsiderar a diferença na variância, já que ela foi controlada pela média que tiramos dos cômodos. O ponto é perceber que de modo geral a casa está mais quente que o ambiente externo.

### Correlação entre as variáveis

Por fim, a matriz de correlação dá uma primeira ideia de quais variáveis se relacionam com o consumo dos eletrodomésticos. As variáveis aleatórias `rv1` e `rv2` servem de referência para correlações espúrias.

In [15]:
corr = original_df.drop(columns='date').corr()
fig = px.imshow(corr, color_continuous_scale='RdBu_r', zmin=-1, zmax=1, aspect='auto',
                title='Matriz de correlação de Pearson')
fig.update_layout(height=800)
fig.show()

Temperaturas tem forte correlação positiva entre sí e umidades também. A umidade do lado de fora da casa (RH6) está negativamente correlacionada com as temperaturas dentro de cada - se chove, a temperatura dentro de casa abaixa. O mesmo em um grau menor com a umidade da estação de observação.

## Preprocessing

Nesta seção construímos os conjuntos de treino, validação e teste **comuns** a todos os modelos, para que os resultados sejam comparáveis.

__Variáveis resposta.__ A tarefa pede a previsão do consumo dos eletrodomésticos (`Appliances`, renomeada para `appliances`) com horizontes de 10 e 60 minutos. Como os dados têm resolução de 10 minutos, as respostas são o próprio `appliances` deslocado 1 e 6 passos à frente:

- `y_10` $= y_{t+1}$, consumo daqui a 10 minutos;
- `y_60` $= y_{t+6}$, consumo daqui a 60 minutos.

Em cada linha, as variáveis explicativas são as observadas no instante $t$ (incluindo o próprio `appliances`, que é a parte autorregressiva) e as respostas são valores futuros.

__Variáveis explicativas.__ Renomeamos as colunas para nomes mais intuitivos (mapeamento completo em `COLUMN_NAMES`): `<cômodo>_temp` e `<cômodo>_hum` para os cômodos, `outside_north_temp`/`outside_north_hum` para o sensor do lado de fora do prédio (`T6`/`RH_6`) e `weather_*` para a estação meteorológica de Chièvres. Removemos `rv1` e `rv2`, que são ruído por construção (foram incluídas no dataset original justamente para testar seleção de variáveis). Como a EDA mostrou um padrão diário forte e diferenças entre os dias da semana, adicionamos:

- `hour_sin`, `hour_cos`: hora do dia em codificação cíclica (23h50 fica próximo de 00h00);
- `dow_0`, ..., `dow_6`: dia da semana em one-hot (0 = segunda-feira).

__Dados meteorológicos.__ As variáveis `weather_*` vêm de observações horárias da estação de Chièvres, interpoladas linearmente para a resolução de 10 minutos. Assim, o valor às 17h10 já mistura a observação das 18h, que ainda não estaria disponível às 17h10. Para evitar esse vazamento, usamos em cada instante o valor da última hora cheia (preenchimento para frente). Esse problema foi identificado na implementação do MLP e agora é corrigido aqui, para todos os modelos.

In [16]:
HORIZONS = {'y_10': 1, 'y_60': 6}  # horizonte em passos de 10 minutos
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15  # teste = restante

COLUMN_NAMES = {
    'Appliances': 'appliances',
    'T1': 'kitchen_temp', 'RH_1': 'kitchen_hum',
    'T2': 'living_room_temp', 'RH_2': 'living_room_hum',
    'T3': 'laundry_temp', 'RH_3': 'laundry_hum',
    'T4': 'office_temp', 'RH_4': 'office_hum',
    'T5': 'bathroom_temp', 'RH_5': 'bathroom_hum',
    'T6': 'outside_north_temp', 'RH_6': 'outside_north_hum',
    'T7': 'ironing_room_temp', 'RH_7': 'ironing_room_hum',
    'T8': 'teen_room_temp', 'RH_8': 'teen_room_hum',
    'T9': 'parents_room_temp', 'RH_9': 'parents_room_hum',
    'T_out': 'weather_temp', 'RH_out': 'weather_hum',
    'Press_mm_hg': 'weather_pressure', 'Windspeed': 'weather_windspeed',
    'Visibility': 'weather_visibility', 'Tdewpoint': 'weather_dewpoint',
}

df = original_df.drop(columns=['date', 'rv1', 'rv2']).rename(columns=COLUMN_NAMES)

WEATHER_COLS = [c for c in df.columns if c.startswith('weather_')]
df[WEATHER_COLS] = df.loc[df.index.minute == 0, WEATHER_COLS].reindex(df.index, method='ffill')

minutes = df.index.hour * 60 + df.index.minute
df['hour_sin'] = np.sin(2 * np.pi * minutes / 1440)
df['hour_cos'] = np.cos(2 * np.pi * minutes / 1440)
df = df.join(pd.get_dummies(df.index.day_of_week, prefix='dow', dtype=int).set_index(df.index))

df.columns.tolist()

['appliances',
 'lights',
 'kitchen_temp',
 'kitchen_hum',
 'living_room_temp',
 'living_room_hum',
 'laundry_temp',
 'laundry_hum',
 'office_temp',
 'office_hum',
 'bathroom_temp',
 'bathroom_hum',
 'outside_north_temp',
 'outside_north_hum',
 'ironing_room_temp',
 'ironing_room_hum',
 'teen_room_temp',
 'teen_room_hum',
 'parents_room_temp',
 'parents_room_hum',
 'weather_temp',
 'weather_pressure',
 'weather_hum',
 'weather_windspeed',
 'weather_visibility',
 'weather_dewpoint',
 'hour_sin',
 'hour_cos',
 'dow_0',
 'dow_1',
 'dow_2',
 'dow_3',
 'dow_4',
 'dow_5',
 'dow_6']

### Divisão em treino, validação e teste

A divisão é __temporal__ (sem embaralhamento): treinamos no passado e avaliamos no futuro, como aconteceria em produção. Embaralhar as linhas vazaria informação, já que observações vizinhas de 10 em 10 minutos são quase idênticas.

Usamos 70% / 15% / 15%, com os cortes arredondados para a meia-noite, de forma que cada conjunto contenha apenas dias completos. Isso dá ~95 dias de treino e ~3 semanas para validação e para teste. Preferimos 15% a 10% na validação porque 10% seriam só 2 semanas, e o consumo médio semanal varia bastante (de ~75 a ~120 Wh), o que deixaria a escolha de hiperparâmetros e o early stopping sensíveis a uma ou duas semanas atípicas.

As respostas são criadas __dentro__ de cada conjunto. Assim, as últimas 6 linhas de cada conjunto (cuja resposta cairia no conjunto seguinte) são descartadas, e nenhum valor de validação ou teste é usado como resposta no treino.

In [17]:
n = len(df)
train_end = df.index[int(n * TRAIN_FRAC)].normalize()
val_end = df.index[int(n * (TRAIN_FRAC + VAL_FRAC))].normalize()


def make_targets(part):
    part = part.copy()
    for name, steps in HORIZONS.items():
        part[name] = part['appliances'].shift(-steps)
    return part.dropna(subset=list(HORIZONS)).astype({name: int for name in HORIZONS})


splits = {
    'train': make_targets(df[df.index < train_end]),
    'val': make_targets(df[(df.index >= train_end) & (df.index < val_end)]),
    'test': make_targets(df[df.index >= val_end]),
}

pd.DataFrame({
    name: {
        'início': s.index.min(),
        'fim': s.index.max(),
        'dias': (s.index.max() - s.index.min()).ceil('D').days,
        'n': len(s),
        'fração': round(len(s) / sum(len(p) for p in splits.values()), 3),
    }
    for name, s in splits.items()
}).T

,início,fim,dias,n,fração
train,2016-01-11 17:00:00,2016-04-15 22:50:00,96,13716,0.696
val,2016-04-16 00:00:00,2016-05-06 22:50:00,21,3018,0.153
test,2016-05-07 00:00:00,2016-05-27 17:00:00,21,2983,0.151


In [18]:
daily_split = pd.concat({name: s['appliances'].resample('D').mean() for name, s in splits.items()},
                        names=['conjunto', 'date']).reset_index()

fig = px.line(daily_split, x='date', y='appliances', color='conjunto',
              title='Consumo médio diário dos eletrodomésticos por conjunto',
              labels={'appliances': 'Wh', 'date': ''})
fig.show()

### Padronização

Padronizamos as variáveis contínuas com o `StandardScaler` do scikit-learn, ajustado __somente no treino__; usar estatísticas de validação ou teste seria vazamento de informação. A função `standardize` recebe as linhas de treino como argumento, o que permite reajustar a padronização em cada dobra da validação temporal (seção "Mudança de distribuição e representação das variáveis"). As variáveis cíclicas e one-hot não são padronizadas.

O ARX usa `standardize` e prevê diretamente em Wh. As redes neurais montam suas próprias entradas (com `appliances` em `log1p`) e ajustam um `StandardScaler` próprio no treino, também descrito nas seções de cada modelo.

In [19]:
TARGETS = list(HORIZONS)
CALENDAR_PREFIXES = ('hour_', 'dow_')


def standardize(frame, train_index):
    """Padroniza as colunas contínuas com média e desvio-padrão das linhas `train_index`.

    Valores ausentes são ignorados no ajuste e preservados na transformação.
    """
    cols = [c for c in frame.columns if not c.startswith(CALENDAR_PREFIXES)]
    scaler = StandardScaler().fit(frame.loc[train_index, cols])
    scaled = frame.copy()
    scaled[cols] = scaler.transform(frame[cols])
    return scaled, scaler


scaled_df, _ = standardize(df, splits['train'].index)

check_cols = ['appliances', 'kitchen_temp', 'weather_temp']
pd.concat({name: scaled_df.loc[s.index, check_cols].agg(['mean', 'std']) for name, s in splits.items()}).round(2)

appliances  kitchen_temp  weather_temp
train mean        0.00          0.00         -0.00
      std         1.00          1.00          1.00
val   mean       -0.04          0.75          0.61
      std         0.88          0.75          1.13
test  mean       -0.02          2.55          2.11
      std         0.85          0.66          1.10

Como esperado, o treino tem média 0 e desvio 1, enquanto validação e teste se afastam disso: o período de teste (maio) é mais quente que o de treino (janeiro a abril). Os modelos terão que extrapolar para temperaturas pouco vistas no treino, algo a lembrar na seção "Quando o modelo de aprendizagem profunda falha?".

### Referência: previsão por persistência

A previsão por persistência ($\hat{y}_{t+h} = y_t$) é a referência do MASE. Definimos

$$\text{MASE} = \frac{\text{MAE}_{\text{modelo}}}{\text{MAE}_{\text{persistência, treino}}},$$

calculando o denominador no conjunto de treino e para o mesmo horizonte do modelo. Um MASE < 1 indica que o modelo supera a persistência.

In [20]:
def persistence_metrics(part):
    rows = {}
    for name in HORIZONS:
        err = part[name] - part['appliances']
        rows[name] = {'MAE': err.abs().mean(), 'RMSE': np.sqrt((err ** 2).mean())}
    return pd.DataFrame(rows).T


MASE_SCALE = persistence_metrics(splits['train'])['MAE']

pd.concat({name: persistence_metrics(s) for name, s in splits.items()}, names=['conjunto', 'horizonte']).round(2)

MAE    RMSE
conjunto horizonte               
train    y_10       30.83   74.12
         y_60       60.29  123.95
val      y_10       26.85   67.60
         y_60       53.29  113.80
test     y_10       26.51   66.46
         y_60       46.02  101.29

### Representação das variáveis de temperatura e umidade

As temperaturas e umidades (dos cômodos, do sensor externo e da estação, além do ponto de orvalho) têm forte componente sazonal: seus níveis sobem ou descem ao longo dos meses. Definimos aqui duas representações para essas variáveis:

- `niveis`: os valores originais. É a representação usada na rodada 1 de todos os modelos.
- `variacoes`: para cada variável, a variação em 10 minutos ($x_t - x_{t-1}$), a variação em 60 minutos ($x_t - x_{t-6}$) e o desvio em relação à média das últimas 24 h ($x_t - \bar{x}_{t-143:t}$). Todas usam apenas valores passados.

A escolha entre as duas é feita na seção "Mudança de distribuição e representação das variáveis", sem usar o conjunto de teste. As demais variáveis (luzes, pressão, vento, visibilidade e calendário) não mudam.

In [21]:
DAY = 144  # passos de 10 minutos em 24 h
TEMP_HUM_COLS = [c for c in df.columns if c.endswith(('_temp', '_hum')) or c == 'weather_dewpoint']
REPRESENTATIONS = ('niveis', 'variacoes')


def apply_representation(frame, representation):
    """Substitui as colunas de temperatura e umidade de `frame` pela representação escolhida."""
    if representation == 'niveis':
        return frame
    if representation != 'variacoes':
        raise ValueError(f'Representação desconhecida: {representation}')

    cols = [c for c in TEMP_HUM_COLS if c in frame.columns]
    raw = frame[cols]
    changes = pd.concat({
        'd10': raw.diff(1),
        'd60': raw.diff(6),
        'dev24h': raw - raw.rolling(DAY).mean(),
    }, axis=1)
    changes.columns = [f'{col}_{kind}' for kind, col in changes.columns]
    return frame.drop(columns=cols).join(changes)


print(f'{len(TEMP_HUM_COLS)} variáveis de temperatura e umidade')
print('Colunas em "variacoes":', apply_representation(df, 'variacoes').shape[1], 'contra', df.shape[1], 'em "niveis"')

21 variáveis de temperatura e umidade
Colunas em "variacoes": 77 contra 35 em "niveis"


### Funções de avaliação compartilhadas

Todos os modelos são avaliados da mesma forma. As previsões de cada modelo ficam em uma tabela longa (`long_predictions`), com uma linha por instante de origem, horizonte, conjunto, modelo, semente e rodada. A partir dela calculamos MAE, RMSE e MASE (`regression_metrics`), resumimos as sementes (média e desvio-padrão amostral) e fazemos os diagnósticos por hora e dos resíduos. A persistência e o ARX entram na mesma tabela, então todas as comparações os incluem.

As previsões são exportadas em CSV para `outputs/` e os padronizadores das redes em `artifacts/` (ambas as pastas ficam fora do git).

In [22]:
PROJECT_ROOT = DATAPATH.resolve().parent.parent
OUTPUTS_PATH, ARTIFACTS_PATH = PROJECT_ROOT / 'outputs', PROJECT_ROOT / 'artifacts'

BASE_LABELS = {'persistencia': 'Persistência', 'arx': 'ARX', 'arx_mlp': 'ARX + MLP', 'mlp': 'MLP',
               'cnn_1d': 'CNN 1D', 'gru': 'GRU', 'lstm': 'LSTM'}
BASE_COLORS = {'persistencia': '#77879A', 'arx': '#C08A1E', 'arx_mlp': '#7A5C12', 'mlp': '#176C82',
               'cnn_1d': '#B34A23', 'gru': '#5B3F8C', 'lstm': '#9A7BC4'}
VARIANT_LABELS = {'_res': ' (residual)', '_lag24': ' (24 h)', '_res_lag24': ' (residual + 24 h)'}

MODEL_LABELS, MODEL_COLORS = {}, {}
for key, label in BASE_LABELS.items():
    variants = VARIANT_LABELS if key in ('mlp', 'cnn_1d') else {}
    for suffix, extra in {'': '', **variants}.items():
        MODEL_LABELS[key + suffix] = label + extra
        MODEL_COLORS[label + extra] = BASE_COLORS[key]
MODEL_ORDER = list(MODEL_LABELS.values())

PREDICTIONS = {}  # (modelo, rodada) -> tabela longa de previsões


def regression_metrics(y_true, y_pred, horizon):
    """MAE, RMSE e MASE em Wh; o MASE é relativo ao MAE da persistência no treino."""
    err = np.asarray(y_true) - np.asarray(y_pred)
    mae = np.abs(err).mean()
    return {'MAE': mae, 'RMSE': np.sqrt((err ** 2).mean()), 'MASE': mae / MASE_SCALE[horizon]}


def long_predictions(model_name, rodada, preds, seed=0, parts=splits):
    """Tabela longa a partir de `preds` = {conjunto: {horizonte: Series de previsões em Wh}}."""
    tables = []
    for split_name, by_horizon in preds.items():
        for horizon, pred in by_horizon.items():
            minutes = 10 * HORIZONS[horizon]
            tables.append(pd.DataFrame({
                'origem': pred.index,
                'timestamp': pred.index + pd.Timedelta(minutes=minutes),
                'horizonte_min': minutes,
                'split': split_name,
                'modelo': model_name,
                'seed': seed,
                'rodada': rodada,
                'y_true': parts[split_name].loc[pred.index, horizon].to_numpy(),
                'y_pred': pred.to_numpy(),
            }))
    return pd.concat(tables, ignore_index=True)


def predictions_table(rodada=None, split=None):
    """Junta as previsões registradas, opcionalmente filtrando por rodada e conjunto."""
    table = pd.concat(PREDICTIONS.values(), ignore_index=True)
    if rodada is not None:
        table = table[table['rodada'] == rodada]
    if split is not None:
        table = table[table['split'] == split]
    return table


def metrics_by_seed(table):
    """MAE, RMSE e MASE para cada rodada, modelo, conjunto, horizonte e semente."""
    rows = []
    keys = ['rodada', 'modelo', 'split', 'horizonte_min', 'seed']
    for values, part in table.groupby(keys, sort=False):
        rows.append({**dict(zip(keys, values)),
                     **regression_metrics(part['y_true'], part['y_pred'], f'y_{values[3]}')})
    return pd.DataFrame(rows)


def summarize_seeds(metrics):
    """Média e desvio-padrão amostral das métricas entre sementes."""
    summary = (metrics.groupby(['rodada', 'modelo', 'split', 'horizonte_min'], sort=False)[['MAE', 'RMSE', 'MASE']]
               .agg(['mean', 'std'])
               .rename(columns={'mean': 'média', 'std': 'DP'}, level=1))
    return summary.rename(index=MODEL_LABELS, level='modelo')


def export_predictions(model_name, rodada):
    """Salva as previsões de um modelo e rodada em `outputs/preds_<modelo>_rodada<k>.csv`."""
    OUTPUTS_PATH.mkdir(exist_ok=True)
    path = OUTPUTS_PATH / f'preds_{model_name}_rodada{rodada}.csv'
    PREDICTIONS[(model_name, rodada)].to_csv(path, index=False)
    return path


def export_scaler(scaler, model_name, rodada):
    """Salva o padronizador ajustado no treino em `artifacts/scaler_<modelo>_rodada<k>.joblib`."""
    ARTIFACTS_PATH.mkdir(exist_ok=True)
    path = ARTIFACTS_PATH / f'scaler_{model_name}_rodada{rodada}.joblib'
    joblib.dump(scaler, path)
    return path


persistence_preds = {name: {h: s['appliances'] for h in TARGETS} for name, s in splits.items()}
for rodada in (1, 2):
    PREDICTIONS[('persistencia', rodada)] = long_predictions('persistencia', rodada, persistence_preds)

summarize_seeds(metrics_by_seed(predictions_table(rodada=1))).round(3)

MAE         RMSE       MASE    
                                          média  DP    média  DP  média  DP
rodada modelo       split horizonte_min                                    
1      Persistência train 10             30.833 NaN   74.119 NaN  1.000 NaN
                          60             60.285 NaN  123.948 NaN  1.000 NaN
                    val   10             26.849 NaN   67.597 NaN  0.871 NaN
                          60             53.290 NaN  113.803 NaN  0.884 NaN
                    test  10             26.510 NaN   66.463 NaN  0.860 NaN
                          60             46.024 NaN  101.289 NaN  0.763 NaN

Funções de gráficos e diagnósticos usadas nas comparações entre modelos. Para as redes, as métricas por hora e o resíduo médio são calculados primeiro em cada semente e depois resumidos pela média entre sementes.

In [23]:
def clean_facet_titles(fig):
    """Troca títulos de facetas do tipo 'horizonte=10 min' por '10 min'."""
    return fig.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split('=')[-1]))


def present_models(table):
    return [MODEL_LABELS[m] for m in MODEL_LABELS if m in set(table['modelo'])]


def plot_metric_bars(table, title):
    """Barras de MAE, RMSE e MASE por horizonte e modelo (média ± DP entre sementes) de uma rodada e conjunto."""
    summary = summarize_seeds(metrics_by_seed(table)).droplevel(['rodada', 'split'])
    metrics = ('MAE', 'RMSE', 'MASE')
    fig = make_subplots(rows=1, cols=3, subplot_titles=metrics)
    for column, metric in enumerate(metrics, start=1):
        for label in present_models(table):
            part = summary.loc[label]
            fig.add_bar(x=[f'{m} min' for m in part.index], y=part[(metric, 'média')],
                        error_y={'type': 'data', 'array': part[(metric, 'DP')].fillna(0)},
                        name=label, legendgroup=label, showlegend=column == 1,
                        marker_color=MODEL_COLORS[label], row=1, col=column)
    fig.update_layout(title=title, barmode='group', template='plotly_white')
    fig.update_yaxes(rangemode='tozero')
    fig.update_yaxes(title_text='Erro (Wh)', row=1, col=1)
    fig.update_yaxes(matches='y', row=1, col=2)
    fig.update_yaxes(title_text='Erro escalonado', row=1, col=3)
    fig.add_hline(y=1, line_dash='dash', line_color='#536776', layer='below', row=1, col=3)
    fig.show()


def with_errors(table):
    out = table.assign(
        hora=table['timestamp'].dt.hour,
        residuo_wh=table['y_true'] - table['y_pred'],
        horizonte=table['horizonte_min'].astype(str) + ' min',
        modelo=table['modelo'].map(MODEL_LABELS),
    )
    return out.assign(erro_abs_wh=out['residuo_wh'].abs())


def plot_by_hour(table, column, title, ylabel):
    """Média por hora do instante previsto (por semente e depois entre sementes) de `column`."""
    data = (with_errors(table)
            .groupby(['modelo', 'horizonte', 'hora', 'seed'])[column].mean()
            .groupby(['modelo', 'horizonte', 'hora']).mean()
            .reset_index())
    fig = px.line(data.sort_values('hora'), x='hora', y=column, color='modelo', line_dash='modelo',
                  facet_col='horizonte', markers=True,
                  category_orders={'horizonte': ['10 min', '60 min'], 'modelo': MODEL_ORDER},
                  color_discrete_map=MODEL_COLORS, template='plotly_white', title=title,
                  labels={'hora': 'Hora do instante previsto', column: ylabel, 'modelo': 'Modelo'})
    clean_facet_titles(fig)
    fig.update_xaxes(tickvals=[0, 4, 8, 12, 16, 20, 23])
    if column == 'residuo_wh':
        fig.add_hline(y=0, line_dash='dash', line_color='#536776')
    else:
        fig.update_yaxes(rangemode='tozero')
    fig.show()
    return data


def mean_bias(table):
    """Resíduo médio (observado − previsto) em Wh por modelo e horizonte."""
    return (with_errors(table)
            .groupby(['modelo', 'horizonte', 'seed'])['residuo_wh'].mean()
            .groupby(['modelo', 'horizonte']).mean()
            .unstack('horizonte')
            .reindex(present_models(table)))


def residual_diagnostics(table, title):
    """ACF média dos resíduos (até 24 h) e teste de Ljung–Box com origens espaçadas pelo horizonte."""
    acf_parts, lb_records = [], []
    for (model, minutes, seed), part in with_errors(table).groupby(['modelo', 'horizonte_min', 'seed'], sort=False):
        residuals = part.sort_values('origem')['residuo_wh'].to_numpy()
        values = acf(residuals, nlags=DAY, fft=True)
        acf_parts.append(pd.DataFrame({'modelo': model, 'horizonte': f'{minutes} min',
                                       'lag_horas': np.arange(1, len(values)) / 6, 'autocorrelacao': values[1:]}))
        lags_one_day = 24 * 60 // minutes
        lb = acorr_ljungbox(residuals[::minutes // 10], lags=[lags_one_day], return_df=True)
        lb_records.append({'modelo': model, 'horizonte_min': minutes, 'seed': seed,
                           'lags': lags_one_day, 'p_valor': lb['lb_pvalue'].iloc[0]})

    acf_mean = (pd.concat(acf_parts).groupby(['modelo', 'horizonte', 'lag_horas'], as_index=False)
                .agg(autocorrelacao=('autocorrelacao', 'mean')))
    fig = px.line(acf_mean.sort_values('lag_horas'), x='lag_horas', y='autocorrelacao', color='modelo',
                  line_dash='modelo', facet_col='horizonte',
                  category_orders={'horizonte': ['10 min', '60 min'], 'modelo': MODEL_ORDER},
                  color_discrete_map=MODEL_COLORS, template='plotly_white', title=title,
                  labels={'lag_horas': 'Defasagem (horas)', 'autocorrelacao': 'Autocorrelação dos resíduos',
                          'modelo': 'Modelo'})
    clean_facet_titles(fig)
    fig.add_hline(y=0, line_dash='dash', line_color='#536776')
    fig.update_xaxes(tickvals=[0, 4, 8, 12, 16, 20, 24])
    fig.show()

    lb = (pd.DataFrame(lb_records).groupby(['horizonte_min', 'modelo'], as_index=False)
          .agg(n_series=('p_valor', 'size'), lags=('lags', 'first'), p_min=('p_valor', 'min'), p_max=('p_valor', 'max')))
    return lb.assign(p_min=lb['p_min'].map('{:.2e}'.format), p_max=lb['p_max'].map('{:.2e}'.format))


def plot_residuals_vs_fitted(table, title):
    """Resíduos contra previsões (semente 0 de cada modelo)."""
    data = with_errors(table[table['seed'] == 0])
    fig = px.scatter(data, x='y_pred', y='residuo_wh', color='modelo', facet_col='horizonte', facet_row='modelo',
                     category_orders={'horizonte': ['10 min', '60 min'], 'modelo': MODEL_ORDER},
                     color_discrete_map=MODEL_COLORS, opacity=0.3, render_mode='webgl', template='plotly_white',
                     title=title, labels={'y_pred': 'Previsão (Wh)', 'residuo_wh': 'Resíduo (Wh)', 'modelo': 'Modelo'})
    clean_facet_titles(fig)
    fig.update_traces(marker_size=3)
    fig.update_layout(height=250 * data['modelo'].nunique(), showlegend=False)
    fig.show()


def cost_summary(runs_by_model):
    """Parâmetros e tempos médios de treino e inferência no teste (média e DP entre sementes)."""
    return (pd.concat(runs_by_model, names=['modelo', 'seed'])
            .groupby(level='modelo', sort=False)
            .agg(parametros=('parametros', 'first'),
                 treino_media_s=('tempo_treino_s', 'mean'), treino_dp_s=('tempo_treino_s', 'std'),
                 inferencia_media_ms=('tempo_inferencia_teste_ms', 'mean'),
                 inferencia_dp_ms=('tempo_inferencia_teste_ms', 'std')))

__Convenções para todos os modelos.__ Como todas as etapas acima são determinísticas e baratas, os conjuntos não são salvos em disco: basta rodar esta seção para obter exatamente os mesmos dados. Defasagens e janelas são construídas sobre a série completa e só depois as linhas de cada conjunto são selecionadas: validação e teste podem usar histórico do conjunto anterior (apenas entradas passadas, sem vazamento), e todos os modelos são avaliados exatamente nas mesmas linhas de validação e teste que a persistência. Só as primeiras linhas do treino, sem histórico completo, são descartadas.

## Baselines

### Autorregressivo

#### Autocorrelação e escolha das defasagens

Antes de ajustar o modelo, olhamos a função de autocorrelação (ACF) e a autocorrelação parcial (PACF) de `appliances` para escolher o número de defasagens $p$:

- a __ACF__ mede a correlação entre $y_t$ e $y_{t-k}$, e revela padrões sazonais;
- a __PACF__ mede a correlação entre $y_t$ e $y_{t-k}$ descontado o efeito das defasagens intermediárias. Em um AR($p$), a PACF é aproximadamente zero após a defasagem $p$, o que a torna a principal ferramenta para escolher $p$.

Calculamos ambas __somente no treino__, para que a escolha de $p$ não use informação de validação ou teste. A faixa cinza é o intervalo de 95% sob a hipótese de ruído branco ($\pm 1{,}96/\sqrt{n}$).

In [24]:
ACF_LAGS, PACF_LAGS = 2 * 144, 48  # 2 dias e 8 horas, em passos de 10 minutos

train_appliances = splits['train']['appliances']
conf = 1.96 / np.sqrt(len(train_appliances))

fig = make_subplots(rows=2, cols=1, vertical_spacing=0.12,
                    subplot_titles=(f'ACF ({ACF_LAGS} defasagens = 2 dias)', f'PACF ({PACF_LAGS} defasagens = 8 horas)'))
for row, values in enumerate([acf(train_appliances, nlags=ACF_LAGS), pacf(train_appliances, nlags=PACF_LAGS)], start=1):
    fig.add_trace(go.Bar(x=np.arange(1, len(values)), y=values[1:], showlegend=False), row=row, col=1)
    fig.add_hrect(y0=-conf, y1=conf, fillcolor='gray', opacity=0.2, line_width=0, row=row, col=1)
fig.add_vline(x=144, line_dash='dash', annotation_text='24 h', row=1, col=1)
fig.update_xaxes(title_text='Defasagem (passos de 10 min)')
fig.update_layout(height=650, title='Autocorrelação de appliances (treino)')
fig.show()

__Interpretação.__

- __PACF:__ a defasagem 1 domina (≈ 0,76); as seguintes são pequenas (|PACF| ≤ 0,13) e praticamente somem após ~10 defasagens (1h40). Com ~13.700 observações a faixa de 95% é muito estreita (± 0,017), então defasagens estatisticamente significativas podem ser irrelevantes na prática. Por isso, $p$ será escolhido pelo MAE de validação numa grade pequena, $p \in \{1, 3, 6, 9, 12\}$.
- __ACF:__ decai rápido nas primeiras defasagens e depois oscila, com pico em 144 passos (24 h, ≈ 0,20) e vale em ~72 passos (12 h, ≈ −0,11). Isso confirma a sazonalidade diária vista na EDA. As variáveis `hour_sin`/`hour_cos` já capturam parte desse padrão, mas vale testar também a defasagem de 24 h ($y_{t-144}$) como variável adicional.

#### Modelo ARX(p, q)

Para cada horizonte $h$ ajustamos uma regressão linear separada (estratégia direta):

$$\hat{y}_{t+h} = c + \sum_{i=0}^{p-1} \phi_i \, y_{t-i} + \sum_{j=0}^{q} \beta_j^\top x_{t-j} + \gamma^\top z_t \; \left[ + \, \delta \, y_{t+h-144} \right]$$

em que $y$ é `appliances`, $x$ são as demais variáveis medidas (luzes, temperaturas, umidades e clima), $z$ são as variáveis de calendário (`hour_*`, `dow_*`), que entram só no instante $t$ por serem determinísticas, e o último termo, opcional, é o consumo no mesmo horário do alvo no dia anterior.

Decisões:

- __Histórico entre conjuntos.__ As defasagens seguem a convenção definida no pré-processamento: são construídas sobre a série completa e depois selecionadas por conjunto.
- __Padronização.__ As entradas são padronizadas com `standardize`, ajustada nas linhas de treino recebidas. Deslocar a série no tempo comuta com a padronização coluna a coluna, então as defasagens herdam a escala da variável original.
- __Resposta em Wh.__ No Ridge, multiplicar a resposta por uma constante multiplica os coeficientes pela mesma constante (e o intercepto absorve a média), então as previsões não dependem da escala da resposta. Ajustamos o modelo diretamente em Wh.
- __Ridge.__ As variáveis são fortemente colineares (temperaturas dos cômodos entre si, defasagens consecutivas, defasagens das exógenas). A penalização estabiliza os coeficientes, e com $\alpha \to 0$ recuperamos o OLS. Também removemos a dummy `dow_6` (domingo): com intercepto, as 7 dummies somam 1 e tornariam a matriz singular (armadilha das dummies).
- __Seleção.__ $p$, $q$, a defasagem de 24 h e $\alpha$ são escolhidos pelo MAE de validação, separadamente para cada horizonte.
- __Representação.__ As funções recebem a representação das variáveis de temperatura e umidade; a rodada 1 usa `niveis`.

In [25]:
P_GRID, Q_GRID, LAG24_GRID = [1, 3, 6, 9, 12, 18, 24], [0, 1, 3], [False, True]
ALPHAS = np.logspace(-2, 4, 7)


def make_lagged(frame, cols, lags):
    """Defasagens `{col}_lag{k}` das colunas `cols`, para cada k em `lags`."""
    return pd.DataFrame({f'{col}_lag{k}': frame[col].shift(k) for k in lags for col in cols}, index=frame.index)


def arx_inputs(representation, train_index):
    """Entradas do ARX na representação escolhida, padronizadas com as linhas de treino."""
    return standardize(apply_representation(df, representation), train_index)[0]


def build_arx(inputs, parts, horizon, p, q, lag24=False):
    """Matrizes (X, y em Wh, índice) de cada conjunto em `parts` para o ARX(p, q) no horizonte `horizon`."""
    exog_cols = [c for c in inputs.columns if not c.startswith(CALENDAR_PREFIXES) and c != 'appliances']
    blocks = [inputs,
              make_lagged(inputs, ['appliances'], range(1, p)),
              make_lagged(inputs, exog_cols, range(1, q + 1))]
    if lag24:
        blocks.append(make_lagged(inputs, ['appliances'], [DAY - HORIZONS[horizon]]))
    design = pd.concat(blocks, axis=1).drop(columns='dow_6')  # evita a armadilha das dummies com o intercepto

    sets = {}
    for name, part in parts.items():
        rows = design.loc[part.index].dropna()
        sets[name] = (rows.to_numpy(), part.loc[rows.index, horizon].to_numpy(), rows.index)
    return sets


def arx_grid(representation, parts=splits):
    """MAE de validação (Wh) de cada combinação de p, q, defasagem de 24 h e alpha, por horizonte."""
    inputs = arx_inputs(representation, parts['train'].index)
    records = []
    for horizon, p, q, lag24 in product(TARGETS, P_GRID, Q_GRID, LAG24_GRID):
        sets = build_arx(inputs, parts, horizon, p, q, lag24)
        (X_tr, y_tr, _), (X_va, y_va, _) = sets['train'], sets['val']
        for alpha in ALPHAS:
            pred = Ridge(alpha=alpha).fit(X_tr, y_tr).predict(X_va)
            records.append({'horizonte': horizon, 'p': p, 'q': q, 'lag24': lag24, 'alpha': alpha,
                            'MAE_val': regression_metrics(y_va, pred, horizon)['MAE']})
    return pd.DataFrame(records)


def best_arx(search):
    """Melhor configuração (menor MAE de validação) de cada horizonte."""
    return search.loc[search.groupby('horizonte')['MAE_val'].idxmin()].set_index('horizonte')


def fit_arx(best, representation, parts=splits):
    """Ajusta a melhor configuração de cada horizonte no treino de `parts` e prevê todos os conjuntos."""
    inputs = arx_inputs(representation, parts['train'].index)
    models, preds = {}, {name: {} for name in parts}
    for horizon in TARGETS:
        cfg = best.loc[horizon]
        sets = build_arx(inputs, parts, horizon, int(cfg['p']), int(cfg['q']), bool(cfg['lag24']))
        models[horizon] = Ridge(alpha=cfg['alpha']).fit(*sets['train'][:2])
        for name, (X_s, _, index) in sets.items():
            preds[name][horizon] = pd.Series(models[horizon].predict(X_s), index=index)
    return models, preds


def plot_arx_search(search, title):
    """MAE de validação em função de p, para cada q e com ou sem a defasagem de 24 h (melhor alpha)."""
    by_config = search.groupby(['horizonte', 'p', 'q', 'lag24'], as_index=False)['MAE_val'].min()
    fig = px.line(by_config.assign(q=by_config['q'].astype(str)), x='p', y='MAE_val', color='q', line_dash='lag24',
                  facet_col='horizonte', markers=True, title=title,
                  labels={'MAE_val': 'MAE de validação (Wh)', 'lag24': 'defasagem 24 h'})
    fig.update_yaxes(matches=None, showticklabels=True)
    fig.show()

#### Seleção de hiperparâmetros na validação

In [26]:
arx_search_r1 = arx_grid('niveis')
arx_best_r1 = best_arx(arx_search_r1)
arx_best_r1

,p,q,lag24,alpha,MAE_val
horizonte,,,,,
y_10,12,0,True,100.0,27.445966
y_60,12,0,True,1000.0,42.939072


In [27]:
plot_arx_search(arx_search_r1, 'MAE de validação do ARX, rodada 1 (melhor alpha para cada configuração)')

#### Resultados

Reajustamos a melhor configuração de cada horizonte no treino e registramos as previsões dos três conjuntos na tabela longa. Validação e teste têm exatamente as mesmas linhas para todos os modelos; no treino, o ARX perde as primeiras linhas (sem histórico completo). Os diagnósticos dos resíduos do ARX aparecem junto com os das redes, na seção "Comparação entre os modelos".

In [28]:
_, arx_preds_r1 = fit_arx(arx_best_r1, 'niveis')
PREDICTIONS[('arx', 1)] = long_predictions('arx', 1, arx_preds_r1)
export_predictions('arx', 1)

summarize_seeds(metrics_by_seed(predictions_table(rodada=1))).round(3)

MAE         RMSE       MASE    
                                          média  DP    média  DP  média  DP
rodada modelo       split horizonte_min                                    
1      Persistência train 10             30.833 NaN   74.119 NaN  1.000 NaN
                          60             60.285 NaN  123.948 NaN  1.000 NaN
                    val   10             26.849 NaN   67.597 NaN  0.871 NaN
                          60             53.290 NaN  113.803 NaN  0.884 NaN
                    test  10             26.510 NaN   66.463 NaN  0.860 NaN
                          60             46.024 NaN  101.289 NaN  0.763 NaN
       ARX          train 10             31.646 NaN   66.776 NaN  1.026 NaN
                          60             50.129 NaN   92.532 NaN  0.832 NaN
                    val   10             27.446 NaN   60.583 NaN  0.890 NaN
                          60             42.939 NaN   84.354 NaN  0.712 NaN
                    test  10             27.083 NaN   59.440 NaN  0.878 NaN
                          60             40.270 NaN   78.386 NaN  0.668 NaN

### MLP

Para cada instante de previsão, montamos um vetor de entrada com o consumo atual e os 11 consumos anteriores, todos na escala `log1p`. Acrescentamos a média móvel de 1 hora, o desvio-padrão móvel de 6 horas, as medições dos sensores (na representação escolhida), os dados meteorológicos da última hora cheia e as variáveis de calendário. Opcionalmente (`lag24=True`), acrescentamos o consumo 24 h antes de cada um dos seis instantes previstos, a mesma informação que o ARX usa na defasagem de 24 h.

O MLP prevê seis valores de consumo, um para cada passo de 10 minutos à frente. Na avaliação, usamos o primeiro e o sexto passos, que correspondem aos horizontes de 10 e 60 minutos, e convertemos as previsões de volta para Wh.

#### Entradas e alvos do MLP

In [29]:
MLP_HISTORY = 12


def build_features(frame, L, representation='niveis', lag24=False):
    """Entradas do MLP (histórico em log1p, estatísticas móveis e variáveis atuais) e os seis alvos em log1p."""
    consumption_log = np.log1p(frame['appliances']).rename('y')

    history = make_lagged(consumption_log.to_frame(), ['y'], range(L))
    history['y_mean_1h'] = consumption_log.rolling(6).mean()
    history['y_std_6h'] = consumption_log.rolling(36).std()
    if lag24:
        for k in range(1, 7):
            history[f'y_24h_h{k}'] = consumption_log.shift(DAY - k)  # consumo 24 h antes do instante t+k

    current = apply_representation(frame.drop(columns='appliances'), representation)
    current['is_weekend'] = (frame.index.day_of_week >= 5).astype(int)

    features = pd.concat([history, current], axis=1)
    targets = pd.DataFrame({f'h{h}': consumption_log.shift(-h) for h in range(1, 7)}, index=frame.index)
    return features, targets


mlp_features, mlp_targets = build_features(df, L=MLP_HISTORY)

print('Entradas:', mlp_features.shape)
print('Alvos:', mlp_targets.shape)
print('Linhas com entradas completas:', len(mlp_features.dropna()))

Entradas: (19735, 49)
Alvos: (19735, 6)
Linhas com entradas completas: 19700


#### Divisão temporal e padronização

Os cortes cronológicos de treino, validação e teste são os do pré-processamento (ou os de cada dobra, na validação temporal). Mantemos as observações com entradas completas e com os seis alvos dentro do respectivo conjunto. As primeiras 35 linhas do treino são descartadas porque ainda não há observações suficientes para calcular o desvio-padrão móvel de 6 horas (com `lag24=True`, as primeiras 143).

Ajustamos o `StandardScaler` apenas às entradas do treino e aplicamos a mesma transformação à validação e ao teste. Os alvos permanecem na escala `log1p`. Também guardamos o índice temporal de cada linha para associar as previsões aos horários correspondentes na avaliação. Com `residual=True`, as entradas ganham um segundo array com $\log(1 + y_t)$ sem padronização, usado pela cabeça residual (seção "Rodada 2").

In [30]:
def prepare_mlp(L, representation='niveis', parts=splits, lag24=False, residual=False):
    """Conjuntos {nome: (entradas, alvos log1p, índice)} do MLP e o padronizador ajustado no treino.

    As entradas são o array padronizado ou, com `residual=True`, a tupla (array padronizado, log1p(y_t)).
    """
    features, targets = build_features(df, L, representation, lag24)
    complete = features.join(targets).dropna().index
    rows = {name: part.index.intersection(complete) for name, part in parts.items()}

    scaler = StandardScaler().fit(features.loc[rows['train']])
    sets = {}
    for name, index in rows.items():
        inputs = scaler.transform(features.loc[index]).astype(np.float32)
        if residual:
            inputs = (inputs, features.loc[index, ['y_lag0']].to_numpy(dtype=np.float32, copy=True))
        sets[name] = (inputs, targets.loc[index].to_numpy(dtype=np.float32, copy=True), index)
    return sets, scaler


mlp_sets, _ = prepare_mlp(MLP_HISTORY)

pd.DataFrame.from_dict({name: {'X': inputs.shape, 'y': targets.shape}
                        for name, (inputs, targets, _) in mlp_sets.items()}, orient='index')

,X,y
train,"(13681, 49)","(13681, 6)"
val,"(3018, 49)","(3018, 6)"
test,"(2983, 49)","(2983, 6)"


#### Arquitetura do MLP

O MLP é o baseline não linear do projeto. Para cada instante de previsão, a rede recebe o vetor de entradas e o transforma por meio de duas camadas ocultas. Cada camada usa a ativação ReLU e dropout durante o treinamento.

A camada de saída produz seis valores na escala `log1p`, um para cada passo de 10 minutos à frente. Nesta etapa, passamos quatro exemplos pela rede para conferir as dimensões da saída antes do treinamento.

In [31]:
class MLP(nn.Module):
    def __init__(self, n_in, n_out=6, hidden=128, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_out),
        )

    def forward(self, x):
        return self.net(x)


X_train_mlp, y_train_mlp, _ = mlp_sets['train']
mlp_check = MLP(n_in=X_train_mlp.shape[1], n_out=y_train_mlp.shape[1])
mlp_check.eval()

with torch.no_grad():
    example_predictions = mlp_check(torch.from_numpy(X_train_mlp[:4]))

print('Dimensões da saída:', tuple(example_predictions.shape))
print('Número de parâmetros:', sum(p.numel() for p in mlp_check.parameters()))

Dimensões da saída: (4, 6)
Número de parâmetros: 23686


#### Treinamento das redes

As funções abaixo são usadas pelo MLP, pela CNN 1D e pela GRU. Por padrão, treinamos em lotes de 256 exemplos com o otimizador AdamW e a perda de Huber, calculada sobre os alvos na escala `log1p`. Em comparação com o erro quadrático, a perda de Huber dá menos peso aos erros muito grandes. Os lotes são embaralhados: cada exemplo já carrega seu próprio histórico, então a ordem dos exemplos dentro do treino não importa.

Ao fim de cada época, calculamos a perda no conjunto de parada antecipada (a validação, na divisão oficial). O treinamento dura até 200 épocas e para se essa perda passar 15 épocas consecutivas sem melhorar. Ao final, restauramos os pesos da época com a menor perda. As perdas de treino e de parada antecipada de cada época ficam em `model.history_`, para as curvas de aprendizado.

- `train_network` treina uma rede: o MLP por padrão, ou a criada por `model_factory(entradas, n_saídas, **hiperparâmetros)`. As entradas podem ser um array ou uma tupla de arrays (por exemplo, janela e vetor de variáveis). Opções usadas pelos modelos novos: `loss` (`'huber'` ou `'l1'`), `grad_clip` (norma máxima do gradiente) e `device`.
- `ResidualOutput` é a cabeça residual: recebe $\log(1 + y_t)$ como última entrada e o soma à saída da rede, que passa a prever a variação em relação ao consumo atual.
- `network_grid` treina cada combinação de hiperparâmetros com a semente `0` e ordena pela média dos MAEs de validação de 10 e 60 minutos, em Wh;
- `train_seeds` repete o treinamento da configuração escolhida com várias sementes, registrando parâmetros e tempos de treino e de inferência no teste;
- `nn_predictions` converte as previsões para Wh e seleciona os horizontes avaliados (`output_steps` indica a que passo corresponde cada saída da rede).

In [32]:
LOSSES = {'huber': nn.HuberLoss, 'l1': nn.L1Loss}


def as_tuple(inputs):
    return inputs if isinstance(inputs, tuple) else (inputs,)


def subset(data, mask):
    """Linhas `mask` de um conjunto (entradas, alvos, índice), com entradas simples ou em tupla."""
    inputs, targets, index = data
    picked = tuple(x[mask] for x in as_tuple(inputs))
    return (picked if isinstance(inputs, tuple) else picked[0]), targets[mask], index[mask]


def make_loader(split_data, batch_size, shuffle):
    inputs, targets, _ = split_data
    tensors = [torch.from_numpy(x) for x in as_tuple(inputs)] + [torch.from_numpy(targets)]
    return DataLoader(TensorDataset(*tensors), batch_size=batch_size, shuffle=shuffle)


class ResidualOutput(nn.Module):
    """Soma a última entrada (log1p do consumo atual) à saída da rede: a rede prevê a variação."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, *xs):
        *inputs, base = xs
        return self.model(*inputs) + base


def mlp_model(inputs, n_out, hidden=128, dropout=0.1, residual=False):
    model = MLP(n_in=as_tuple(inputs)[0].shape[1], n_out=n_out, hidden=hidden, dropout=dropout)
    return ResidualOutput(model) if residual else model


def train_network(
    train_data, val_data, *, seed=0, lr=1e-3, batch_size=256, max_epochs=200, patience=15,
    loss='huber', grad_clip=None, device='cpu', model_factory=mlp_model, **model_kwargs,
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    train_loader = make_loader(train_data, batch_size, shuffle=True)
    val_loader = make_loader(val_data, batch_size, shuffle=False)

    inputs, targets, _ = train_data
    model = model_factory(inputs, targets.shape[1], **model_kwargs).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = LOSSES[loss]()

    best_loss = float('inf')
    best_state = None
    best_epoch = 0
    history = []
    started_at = perf_counter()

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss = 0.0
        for *xb, yb in train_loader:
            xb, yb = [x.to(device) for x in xb], yb.to(device)
            optimizer.zero_grad()
            batch_loss = loss_fn(model(*xb), yb)
            batch_loss.backward()
            if grad_clip is not None:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            train_loss += batch_loss.item() * len(yb)

        model.eval()
        with torch.no_grad():
            val_loss = sum(loss_fn(model(*[x.to(device) for x in xb]), yb.to(device)).item() * len(yb)
                           for *xb, yb in val_loader) / len(val_loader.dataset)
        history.append({'epoca': epoch, 'treino': train_loss / len(train_loader.dataset), 'validacao': val_loss})

        if val_loss < best_loss:
            best_loss = val_loss
            best_state = deepcopy(model.state_dict())
            best_epoch = epoch
        elif epoch - best_epoch >= patience:
            break

    model.load_state_dict(best_state)
    model.history_ = pd.DataFrame(history)
    summary = {
        'epocas_executadas': epoch,
        'melhor_epoca': best_epoch,
        'perda_val': best_loss,
        'tempo_treino_s': perf_counter() - started_at,
    }
    return model, summary


def predict_raw(model, inputs):
    """Saída da rede na escala dos alvos (log1p, para os modelos de consumo)."""
    model.eval()
    device = next(model.parameters()).device
    with torch.no_grad():
        return model(*[torch.from_numpy(x).to(device) for x in as_tuple(inputs)]).cpu().numpy()


def predict_wh(model, inputs):
    return np.expm1(predict_raw(model, inputs))


def nn_predictions(model, sets, output_steps=tuple(range(1, 7))):
    """Previsões em Wh dos horizontes avaliados: {conjunto: {horizonte: Series}}."""
    preds = {}
    for name, (inputs, _, index) in sets.items():
        values = predict_wh(model, inputs)
        preds[name] = {h: pd.Series(values[:, output_steps.index(steps)], index=index)
                       for h, steps in HORIZONS.items() if steps in output_steps}
    return preds


def network_grid(prepare, windows, settings_grid, output_steps=tuple(range(1, 7)), **train_kwargs):
    """Treina cada janela L e combinação de hiperparâmetros (semente 0) e ordena pelo MAE médio de validação."""
    started_at = perf_counter()
    records = []
    for L in windows:
        sets = prepare(L)
        for settings in settings_grid:
            model, info = train_network(sets['train'], sets['val'], seed=0, **train_kwargs, **settings)
            val_preds = nn_predictions(model, {'val': sets['val']}, output_steps)['val']
            mae = {h: regression_metrics(splits['val'].loc[p.index, h], p, h)['MAE'] for h, p in val_preds.items()}
            records.append({'L': L, **settings, 'mae_10_wh': mae['y_10'], 'mae_60_wh': mae['y_60'],
                            'mae_medio_wh': (mae['y_10'] + mae['y_60']) / 2,
                            'melhor_epoca': info['melhor_epoca'], 'tempo_treino_s': info['tempo_treino_s']})
        print(f'L = {L}: {len(settings_grid)} configurações concluídas.', flush=True)
    print(f'Tempo total da grade: {perf_counter() - started_at:.1f} s')
    return pd.DataFrame(records).sort_values('mae_medio_wh').reset_index(drop=True)


def best_config(grid, keys):
    """Janela L e hiperparâmetros da melhor linha de `network_grid`."""
    best = grid.iloc[0]
    return int(best['L']), {k: int(best[k]) if k in ('hidden', 'layers') else float(best[k]) for k in keys}


def train_seeds(sets, n_seeds=5, **train_kwargs):
    """Treina a configuração escolhida com as sementes 0 a n_seeds − 1."""
    models, runs = {}, []
    for seed in range(n_seeds):
        model, info = train_network(sets['train'], sets['val'], seed=seed, **train_kwargs)
        inference_start = perf_counter()
        predict_raw(model, sets['test'][0])
        models[seed] = model
        runs.append({'seed': seed, 'parametros': sum(p.numel() for p in model.parameters()), **info,
                     'tempo_inferencia_teste_ms': 1000 * (perf_counter() - inference_start)})
        print(f"Semente {seed}: treino concluído em {info['tempo_treino_s']:.1f} s", flush=True)
    return models, pd.DataFrame(runs).set_index('seed')


def seed_predictions(model_name, rodada, models, sets, output_steps=tuple(range(1, 7))):
    """Tabela longa com as previsões de todas as sementes."""
    return pd.concat([long_predictions(model_name, rodada, nn_predictions(model, sets, output_steps), seed=seed)
                      for seed, model in models.items()], ignore_index=True)


def plot_learning_curves(models, title):
    """Perda de treino e de parada antecipada por época, para cada semente."""
    curves = pd.concat({seed: model.history_ for seed, model in models.items()}, names=['semente', None])
    curves = curves.reset_index('semente').melt(id_vars=['semente', 'epoca'], var_name='conjunto', value_name='perda')
    fig = px.line(curves.astype({'semente': str}), x='epoca', y='perda', color='semente', line_dash='conjunto',
                  template='plotly_white', title=title,
                  labels={'epoca': 'Época', 'perda': 'Perda', 'conjunto': 'Conjunto', 'semente': 'Semente'})
    fig.show()

#### Seleção dos hiperparâmetros do MLP

Testamos três comprimentos de histórico (`L = 6, 12, 36`), duas larguras para as camadas ocultas, três valores de dropout e duas taxas de aprendizado. Para cada `L`, reconstruímos as entradas e ajustamos o padronizador no treino. Usamos a semente `0` em toda a grade e comparamos as configurações nas mesmas datas da validação. O critério de seleção é a média dos MAEs de 10 e 60 minutos, em Wh.

In [33]:
MLP_WINDOWS = (6, 12, 36)
MLP_SETTINGS_GRID = [{'hidden': hidden, 'dropout': dropout, 'lr': lr}
                     for hidden, dropout, lr in product((64, 128), (0.0, 0.1, 0.2), (1e-3, 3e-4))]

mlp_grid_r1 = network_grid(lambda L: prepare_mlp(L, 'niveis')[0], MLP_WINDOWS, MLP_SETTINGS_GRID)
mlp_grid_r1.head(10).round(4)

L = 6: 12 configurações concluídas.
L = 12: 12 configurações concluídas.
L = 36: 12 configurações concluídas.
Tempo total da grade: 564.4 s


,L,hidden,dropout,lr,mae_10_wh,mae_60_wh,mae_medio_wh,melhor_epoca,tempo_treino_s
0,12,128,0.2,0.0010,26.6674,37.3688,32.0181,49,15.6521
1,6,128,0.2,0.0003,26.4091,37.9980,32.2035,85,24.9147
2,12,64,0.2,0.0003,27.3770,37.1986,32.2878,95,22.6743
3,36,64,0.2,0.0003,26.9872,37.6920,32.3396,105,25.3642
4,36,128,0.2,0.0010,27.0718,37.6353,32.3536,50,16.5173
5,12,128,0.2,0.0003,26.8861,37.8708,32.3785,90,26.0009
6,6,128,0.2,0.0010,26.7968,38.0563,32.4266,38,13.1839
7,6,64,0.2,0.0003,27.2286,37.6651,32.4468,97,22.8148
8,12,64,0.1,0.0003,26.9669,38.0548,32.5109,122,27.7365
9,36,64,0.2,0.0010,27.1846,37.9506,32.5676,44,12.1619


#### Treinamento com 5 sementes

Fixamos os hiperparâmetros da configuração com o menor MAE médio na validação e repetimos o treinamento com as sementes de 0 a 4. Isso permite medir quanto o desempenho varia com a inicialização dos pesos e o embaralhamento dos lotes. Em cada execução, usamos a perda de validação para a parada antecipada e restauramos os pesos da melhor época.

In [34]:
mlp_best_L, mlp_best_settings = best_config(mlp_grid_r1, ['hidden', 'dropout', 'lr'])
mlp_final_sets_r1, mlp_scaler_r1 = prepare_mlp(mlp_best_L, 'niveis')

mlp_models_r1, mlp_runs_r1 = train_seeds(mlp_final_sets_r1, **mlp_best_settings)
plot_learning_curves(mlp_models_r1, 'Curvas de aprendizado do MLP (rodada 1)')
mlp_runs_r1.round(4)

Semente 0: treino concluído em 16.9 s
Semente 1: treino concluído em 19.6 s
Semente 2: treino concluído em 14.8 s
Semente 3: treino concluído em 13.4 s
Semente 4: treino concluído em 17.0 s


,parametros,epocas_executadas,melhor_epoca,perda_val,tempo_treino_s,tempo_inferencia_teste_ms
seed,,,,,,
0,23686,64,49,0.0898,16.8750,0.8468
1,23686,76,61,0.0889,19.5593,0.9384
2,23686,58,43,0.0900,14.7838,0.8459
3,23686,50,35,0.0907,13.4227,0.8282
4,23686,64,49,0.0894,16.9925,0.8137


#### Avaliação do MLP

Registramos as previsões das cinco sementes na tabela longa e reportamos, para cada conjunto e horizonte, a média e o desvio-padrão amostral das métricas entre as sementes, ao lado da persistência e do ARX. As previsões e o padronizador ajustado no treino são exportados.

In [35]:
PREDICTIONS[('mlp', 1)] = seed_predictions('mlp', 1, mlp_models_r1, mlp_final_sets_r1)
print('Previsões:', export_predictions('mlp', 1))
print('Padronizador:', export_scaler(mlp_scaler_r1, 'mlp', 1))

display(summarize_seeds(metrics_by_seed(predictions_table(rodada=1))).round(3))
plot_metric_bars(predictions_table(rodada=1, split='test'), 'Persistência, ARX e MLP no teste (rodada 1)')

Previsões: /content/previsao_curto_prazo_consumo_energia_eletrica_MAE5920/outputs/preds_mlp_rodada1.csv
Padronizador: /content/previsao_curto_prazo_consumo_energia_eletrica_MAE5920/artifacts/scaler_mlp_rodada1.joblib


MAE            RMSE          MASE  \
                                          média     DP    média     DP  média   
rodada modelo       split horizonte_min                                         
1      Persistência train 10             30.833    NaN   74.119    NaN  1.000   
                          60             60.285    NaN  123.948    NaN  1.000   
                    val   10             26.849    NaN   67.597    NaN  0.871   
                          60             53.290    NaN  113.803    NaN  0.884   
                    test  10             26.510    NaN   66.463    NaN  0.860   
                          60             46.024    NaN  101.289    NaN  0.763   
       ARX          train 10             31.646    NaN   66.776    NaN  1.026   
                          60             50.129    NaN   92.532    NaN  0.832   
                    val   10             27.446    NaN   60.583    NaN  0.890   
                          60             42.939    NaN   84.354    NaN  0.712   
                    test  10             27.083    NaN   59.440    NaN  0.878   
                          60             40.270    NaN   78.386    NaN  0.668   
       MLP          train 10             28.901  0.511   68.538  1.646  0.937   
                          60             40.432  0.512   93.467  1.527  0.671   
                    val   10             26.783  0.341   65.270  1.149  0.869   
                          60             38.142  0.581   87.411  0.757  0.633   
                    test  10             36.526  3.280   66.073  1.737  1.185   
                          60             52.376  5.593   86.363  4.095  0.869   

                                                
                                            DP  
rodada modelo       split horizonte_min         
1      Persistência train 10               NaN  
                          60               NaN  
                    val   10               NaN  
                          60               NaN  
                    test  10               NaN  
                          60               NaN  
       ARX          train 10               NaN  
                          60               NaN  
                    val   10               NaN  
                          60               NaN  
                    test  10               NaN  
                          60               NaN  
       MLP          train 10             0.017  
                          60             0.008  
                    val   10             0.011  
                          60             0.010  
                    test  10             0.106  
                          60             0.093

## Modelos

### CNN 1D

#### Janelas temporais para a CNN 1D

A CNN 1D recebe uma janela de canais, em que cada canal representa uma variável acompanhada ao longo do tempo, como o consumo em `log1p`, a temperatura ou a iluminação. Para uma previsão feita às 10h com janela de 12 observações, a janela vai das 8h10 às 10h e os seis alvos vão das 10h10 às 11h.

Reaproveitamos as datas e os alvos do MLP. Os canais são padronizados com média e desvio-padrão do treino. As entradas ficam no formato `(amostras, canais, passos de tempo)`, com os passos em ordem cronológica. Na representação `variacoes`, descartamos as primeiras origens do treino cuja janela alcança o início da série (sem média de 24 h).

Opcionalmente, a CNN recebe também um vetor com o consumo 24 h antes de cada instante previsto (`lag24=True`), concatenado após o `Flatten`, e $\log(1 + y_t)$ para a cabeça residual (`residual=True`).

In [36]:
CNN_WINDOW = 12
LAG24_COLS = [f'y_24h_h{k}' for k in range(1, 7)]


def prepare_cnn(L, representation='niveis', parts=splits, lag24=False, residual=False):
    """Janelas (amostras, canais, L) com as mesmas origens e alvos do MLP, e o padronizador dos canais.

    Com `lag24` ou `residual`, as entradas viram a tupla (janelas[, vetor de 24 h][, log1p(y_t)]).
    """
    features, _ = build_features(df, MLP_HISTORY, representation, lag24)
    channels = ['y_lag0'] + [c for c in features.columns if not c.startswith('y_')]
    frame = features[channels]

    base, _ = prepare_mlp(MLP_HISTORY, representation, parts, lag24)
    scaler = StandardScaler().fit(frame.loc[base['train'][2]])
    windows = sliding_window_view(scaler.transform(frame).astype(np.float32), window_shape=L, axis=0)
    window_end_index = frame.index[L - 1:]
    if lag24:
        vector_scaler = StandardScaler().fit(features.loc[base['train'][2], LAG24_COLS])

    sets = {}
    for name, (_, targets, origins) in base.items():
        positions = window_end_index.get_indexer(origins)
        if (positions < 0).any():
            raise ValueError(f'Histórico insuficiente em {name} para L={L}.')
        inputs = [np.ascontiguousarray(windows[positions])]
        if lag24:
            inputs.append(vector_scaler.transform(features.loc[origins, LAG24_COLS]).astype(np.float32))
        if residual:
            inputs.append(features.loc[origins, ['y_lag0']].to_numpy(dtype=np.float32, copy=True))
        complete = np.isfinite(inputs[0]).all(axis=(1, 2))
        if name != 'train' and not complete.all():
            raise ValueError(f'Janelas incompletas em {name} para L={L}.')
        data = (tuple(inputs) if len(inputs) > 1 else inputs[0], targets, origins)
        sets[name] = subset(data, complete)
    return sets, scaler

In [37]:
cnn_sets, _ = prepare_cnn(CNN_WINDOW)

pd.DataFrame.from_dict({name: {'X': inputs.shape, 'y': targets.shape}
                        for name, (inputs, targets, _) in cnn_sets.items()}, orient='index')

,X,y
train,"(13681, 36, 12)","(13681, 6)"
val,"(3018, 36, 12)","(3018, 6)"
test,"(2983, 36, 12)","(2983, 6)"


#### Arquitetura da CNN 1D

Aplicamos duas convoluções temporais com kernel de tamanho 3, stride de 1 e padding de 1. Cada convolução produz 32 canais e preserva o número de passos de tempo. Aplicamos ReLU após cada convolução.

Em seguida, o `Flatten` reúne os 32 canais e os passos de tempo em um vetor por amostra. Usamos dropout de 20% antes da camada linear, que produz as seis previsões na escala `log1p`.

In [38]:
class CNN1D(nn.Module):
    def __init__(self, n_channels, window_length, n_out=6, filters=32, dropout=0.2, n_vector=0):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_channels, filters, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(filters, filters, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten(start_dim=1),
        )
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(filters * window_length + n_vector, n_out)

    def forward(self, x, vector=None):
        features = self.conv(x)
        if vector is not None:
            features = torch.cat([features, vector], dim=1)
        return self.head(self.dropout(features))


def build_cnn_model(inputs, n_out, lag24=False, residual=False, **_):
    windows = as_tuple(inputs)[0]
    n_vector = as_tuple(inputs)[1].shape[1] if lag24 else 0
    model = CNN1D(n_channels=windows.shape[1], window_length=windows.shape[2], n_out=n_out, n_vector=n_vector)
    return ResidualOutput(model) if residual else model


X_train_cnn, y_train_cnn, _ = cnn_sets['train']
cnn_check = build_cnn_model(X_train_cnn, y_train_cnn.shape[1])
cnn_check.eval()

with torch.no_grad():
    cnn_example = cnn_check(torch.from_numpy(X_train_cnn[:4]))

assert tuple(cnn_example.shape) == (4, y_train_cnn.shape[1])

print('Dimensões da saída:', tuple(cnn_example.shape))
print('Número de parâmetros:', sum(p.numel() for p in cnn_check.parameters()))

Dimensões da saída: (4, 6)
Número de parâmetros: 8902


#### Ajuste dos hiperparâmetros da CNN 1D

Usamos o mesmo protocolo do MLP: AdamW, lotes de 256, perda de Huber sobre os seis alvos em `log1p` e parada antecipada na validação. Testamos janelas de 6, 12 e 36 observações e taxas de aprendizado de `0,001` e `0,0003`, com a semente `0` e as mesmas origens de previsão em todas as configurações. Selecionamos a configuração pelo menor MAE médio dos horizontes de 10 e 60 minutos, calculado em Wh na validação.

In [39]:
CNN_WINDOWS = (6, 12, 36)
CNN_SETTINGS_GRID = [{'lr': lr} for lr in (1e-3, 3e-4)]

cnn_grid_r1 = network_grid(lambda L: prepare_cnn(L, 'niveis')[0], CNN_WINDOWS, CNN_SETTINGS_GRID,
                           model_factory=build_cnn_model)
cnn_grid_r1.round(4)

L = 6: 2 configurações concluídas.
L = 12: 2 configurações concluídas.
L = 36: 2 configurações concluídas.
Tempo total da grade: 94.9 s


,L,lr,mae_10_wh,mae_60_wh,mae_medio_wh,melhor_epoca,tempo_treino_s
0,6,0.0003,31.8040,40.9977,36.4008,45,18.1499
1,6,0.0010,31.6644,41.5464,36.6054,26,12.2268
2,12,0.0003,35.7525,41.3856,38.5691,39,19.1113
3,12,0.0010,36.7087,44.4989,40.6038,19,12.1361
4,36,0.0003,41.8064,48.1823,44.9944,19,18.6790
5,36,0.0010,41.9650,48.2309,45.0980,11,14.1840


#### Treinamento final da CNN 1D

Fixamos a configuração com o menor MAE médio na validação e repetimos o treinamento com as sementes de 0 a 4 para medir a variação do desempenho entre execuções. Em cada treino, aplicamos a parada antecipada na validação e restauramos os pesos da melhor época.

In [40]:
cnn_best_L, cnn_best_settings = best_config(cnn_grid_r1, ['lr'])
cnn_final_sets_r1, cnn_scaler_r1 = prepare_cnn(cnn_best_L, 'niveis')
pd.testing.assert_index_equal(cnn_final_sets_r1['test'][2], mlp_final_sets_r1['test'][2])

cnn_models_r1, cnn_runs_r1 = train_seeds(cnn_final_sets_r1, model_factory=build_cnn_model, **cnn_best_settings)
plot_learning_curves(cnn_models_r1, 'Curvas de aprendizado da CNN 1D (rodada 1)')
cnn_runs_r1.round(4)

Semente 0: treino concluído em 18.2 s
Semente 1: treino concluído em 16.1 s
Semente 2: treino concluído em 18.0 s
Semente 3: treino concluído em 11.7 s
Semente 4: treino concluído em 21.1 s


,parametros,epocas_executadas,melhor_epoca,perda_val,tempo_treino_s,tempo_inferencia_teste_ms
seed,,,,,,
0,7750,60,45,0.1057,18.1715,2.9493
1,7750,53,38,0.1169,16.0933,1.9010
2,7750,58,43,0.1097,18.0119,1.7611
3,7750,39,24,0.1194,11.6959,1.7297
4,7750,70,55,0.1206,21.0750,1.7200


#### Avaliação da CNN 1D

Registramos as previsões das cinco sementes na tabela longa e exportamos as previsões e o padronizador dos canais. Antes, conferimos que os alvos de 10 e 60 minutos, convertidos para Wh, coincidem com o consumo observado nas datas correspondentes.

In [41]:
for split_name, (_, targets_log, index) in cnn_final_sets_r1.items():
    np.testing.assert_allclose(np.expm1(targets_log[:, [s - 1 for s in HORIZONS.values()]]),
                               splits[split_name].loc[index, TARGETS].to_numpy(), rtol=1e-6, atol=1e-5)

PREDICTIONS[('cnn_1d', 1)] = seed_predictions('cnn_1d', 1, cnn_models_r1, cnn_final_sets_r1)
print('Previsões:', export_predictions('cnn_1d', 1))
print('Padronizador:', export_scaler(cnn_scaler_r1, 'cnn_1d', 1))

summarize_seeds(metrics_by_seed(predictions_table(rodada=1).query("modelo == 'cnn_1d'"))).round(3)

Previsões: /content/previsao_curto_prazo_consumo_energia_eletrica_MAE5920/outputs/preds_cnn_1d_rodada1.csv
Padronizador: /content/previsao_curto_prazo_consumo_energia_eletrica_MAE5920/artifacts/scaler_cnn_1d_rodada1.joblib


MAE             RMSE           MASE  \
                                    média      DP    média      DP  média   
rodada modelo split horizonte_min                                           
1      CNN 1D train 10             33.843   1.510   78.218   3.236  1.098   
                    60             41.615   0.734   93.643   0.976  0.690   
              val   10             35.040   2.075   76.994   2.286  1.136   
                    60             43.114   1.357   89.585   0.518  0.715   
              test  10             83.555  18.515  122.131  29.129  2.710   
                    60             91.135  11.723  130.203  20.530  1.512   

                                          
                                      DP  
rodada modelo split horizonte_min         
1      CNN 1D train 10             0.049  
                    60             0.012  
              val   10             0.067  
                    60             0.023  
              test  10             0.600  
                    60             0.194

## Comparação entre os modelos (rodada 1)

Comparamos persistência, ARX, MLP e CNN 1D nas mesmas origens do teste. Para as redes, os valores são a média e o desvio-padrão amostral entre cinco sementes.

In [42]:
test_r1 = predictions_table(rodada=1, split='test')

display(summarize_seeds(metrics_by_seed(predictions_table(rodada=1))).round(3))
plot_metric_bars(test_r1, 'Comparação dos modelos no conjunto de teste (rodada 1)')

MAE             RMSE          \
                                          média      DP    média      DP   
rodada modelo       split horizonte_min                                    
1      Persistência train 10             30.833     NaN   74.119     NaN   
                          60             60.285     NaN  123.948     NaN   
                    val   10             26.849     NaN   67.597     NaN   
                          60             53.290     NaN  113.803     NaN   
                    test  10             26.510     NaN   66.463     NaN   
                          60             46.024     NaN  101.289     NaN   
       ARX          train 10             31.646     NaN   66.776     NaN   
                          60             50.129     NaN   92.532     NaN   
                    val   10             27.446     NaN   60.583     NaN   
                          60             42.939     NaN   84.354     NaN   
                    test  10             27.083     NaN   59.440     NaN   
                          60             40.270     NaN   78.386     NaN   
       MLP          train 10             28.901   0.511   68.538   1.646   
                          60             40.432   0.512   93.467   1.527   
                    val   10             26.783   0.341   65.270   1.149   
                          60             38.142   0.581   87.411   0.757   
                    test  10             36.526   3.280   66.073   1.737   
                          60             52.376   5.593   86.363   4.095   
       CNN 1D       train 10             33.843   1.510   78.218   3.236   
                          60             41.615   0.734   93.643   0.976   
                    val   10             35.040   2.075   76.994   2.286   
                          60             43.114   1.357   89.585   0.518   
                    test  10             83.555  18.515  122.131  29.129   
                          60             91.135  11.723  130.203  20.530   

                                          MASE         
                                         média     DP  
rodada modelo       split horizonte_min                
1      Persistência train 10             1.000    NaN  
                          60             1.000    NaN  
                    val   10             0.871    NaN  
                          60             0.884    NaN  
                    test  10             0.860    NaN  
                          60             0.763    NaN  
       ARX          train 10             1.026    NaN  
                          60             0.832    NaN  
                    val   10             0.890    NaN  
                          60             0.712    NaN  
                    test  10             0.878    NaN  
                          60             0.668    NaN  
       MLP          train 10             0.937  0.017  
                          60             0.671  0.008  
                    val   10             0.869  0.011  
                          60             0.633  0.010  
                    test  10             1.185  0.106  
                          60             0.869  0.093  
       CNN 1D       train 10             1.098  0.049  
                          60             0.690  0.012  
                    val   10             1.136  0.067  
                          60             0.715  0.023  
                    test  10             2.710  0.600  
                          60             1.512  0.194

#### MAE e resíduo médio por hora do dia

Agrupamos as previsões pela hora do instante previsto e calculamos o MAE em Wh e o resíduo médio (observado menos previsto) para cada modelo e horizonte. Um resíduo médio positivo indica subestimação e um negativo, superestimação. Para as redes, calculamos primeiro o valor de cada semente em cada hora e depois a média das cinco sementes. A tabela abaixo dos gráficos apresenta o resíduo médio no teste inteiro.

In [43]:
plot_by_hour(test_r1, 'erro_abs_wh', 'MAE no teste por hora do instante previsto (rodada 1)', 'MAE (Wh)')
plot_by_hour(test_r1, 'residuo_wh', 'Resíduo médio no teste por hora do instante previsto (rodada 1)', 'Resíduo médio (Wh)')
mean_bias(test_r1).round(3)

horizonte,10 min,60 min
modelo,,
Persistência,0.013,0.365
ARX,-0.091,-1.046
MLP,-11.003,-16.005
CNN 1D,-63.160,-64.112


#### Diagnóstico dos resíduos

Calculamos a autocorrelação dos resíduos de teste até 24 horas de defasagem; para as redes, a ACF é calculada em cada semente e mostramos a média das curvas. Aplicamos o teste de Ljung–Box a cada série separadamente. Na estratégia direta, previsões consecutivas de 60 minutos compartilham 5 dos 6 passos futuros, o que gera autocorrelação mesmo em um modelo ótimo; por isso, usamos origens espaçadas de 10 minutos no horizonte de 10 minutos e de 60 minutos no horizonte de 60 minutos. A tabela apresenta o menor e o maior valor-p entre as sementes de cada rede. O último gráfico mostra os resíduos contra as previsões (semente 0).

In [44]:
display(residual_diagnostics(test_r1, 'Autocorrelação dos resíduos no teste (rodada 1)'))
plot_residuals_vs_fitted(test_r1, 'Resíduos vs. previsões no teste (rodada 1)')

,horizonte_min,modelo,n_series,lags,p_min,p_max
0,10,ARX,1,144,4.61e-05,4.61e-05
1,10,CNN 1D,5,144,0.00e+00,0.00e+00
2,10,MLP,5,144,0.00e+00,0.00e+00
3,10,Persistência,1,144,5.20e-51,5.20e-51
4,60,ARX,1,24,8.40e-01,8.40e-01
5,60,CNN 1D,5,24,0.00e+00,1.21e-118
6,60,MLP,5,24,1.13e-14,1.84e-03
7,60,Persistência,1,24,1.35e-22,1.35e-22


#### Custo computacional

Comparamos o número de parâmetros e os tempos medidos nas cinco sementes de cada rede. O tempo de inferência corresponde a uma passagem pelas origens do teste, incluindo a conversão das previsões para Wh.

In [45]:
cost_summary({'MLP': mlp_runs_r1, 'CNN 1D': cnn_runs_r1}).round(3)

,parametros,treino_media_s,treino_dp_s,inferencia_media_ms,inferencia_dp_ms
modelo,,,,,
MLP,23686,16.327,2.346,0.855,0.049
CNN 1D,7750,17.010,3.463,2.012,0.529


__Resumo da rodada 1.__ No teste, a persistência teve o menor MAE aos 10 minutos (26,5 Wh), seguida de perto pelo ARX (27,1 Wh); aos 60 minutos, o ARX foi o melhor modelo (40,3 Wh, contra 46,0 Wh da persistência). O MLP (36,5 ± 3,3 e 52,4 ± 5,6 Wh) e a CNN 1D (83,8 ± 18,5 e 91,4 ± 11,9 Wh) ficaram atrás dos dois. Em RMSE, o ARX foi o melhor nos dois horizontes.

O contraste com a validação é o ponto central. Na validação, o MLP teve MAE de 26,8 e 38,1 Wh: empatado com a persistência aos 10 minutos e bem melhor que o ARX aos 60 minutos (42,9 Wh). No teste, a piora das redes vem acompanhada de grande variação entre sementes e de resíduo médio negativo, ou seja, as redes superestimam o consumo (−11 e −16 Wh para o MLP, −63 e −64 Wh para a CNN), enquanto o ARX e a persistência praticamente não têm viés. Esse padrão é investigado na próxima seção.

Obs.: os números das redes diferem levemente dos obtidos na primeira execução deste notebook, em outro computador (por exemplo, MAE de teste do MLP aos 10 minutos de 37,5 Wh), porque o treinamento em CPUs diferentes produz pequenas diferenças numéricas. As entradas e o código de treino são idênticos.

## Mudança de distribuição e representação das variáveis

Na rodada 1, as redes tiveram desempenho bem pior no teste do que na validação, enquanto o ARX manteve o desempenho. A explicação mais provável é uma __mudança na distribuição das entradas__ (*covariate shift*). O teste (maio) é muito mais quente que o treino (janeiro a meados de abril), então boa parte das temperaturas e umidades do teste fica fora da faixa vista no treino; o consumo, por outro lado, tem distribuição parecida nos três períodos. Fora da faixa de treino, uma rede com ReLU não tem informação sobre como se comportar e extrapola de forma arbitrária, e cada semente extrapola de um jeito, o que explica a grande variação entre sementes no teste. O ARX, linear e com coeficientes encolhidos, extrapola de forma mais contida.

__O que não podemos fazer.__ Remover variáveis ou mudar o pré-processamento porque o resultado de teste melhorou seria ajustar o modelo ao conjunto de teste, que deixaria de ser uma estimativa honesta do erro.

__O que fazemos.__ Comparamos as duas representações das variáveis de temperatura e umidade definidas no pré-processamento (`niveis` e `variacoes`) usando apenas dados de treino e validação. A comparação usa validação com janelas expansivas (*forward chaining*): três blocos de 21 dias, cada um avaliado por modelos treinados somente com os dados anteriores ao bloco.

__Regra de decisão, definida antes de ver os resultados das dobras.__ Escolhemos a representação com o menor MASE médio nas dobras, considerando os dois horizontes, o ARX e o MLP (no MLP, a média das sementes). A representação escolhida é usada por todos os modelos na rodada 2 (ARX, MLP, CNN 1D e, futuramente, a GRU). Os hiperparâmetros ficam fixos nos valores escolhidos na rodada 1, para que as dobras comparem apenas a representação. Isso favorece levemente `niveis`, a representação usada para escolhê-los.

__Ressalvas.__ A EDA foi feita sobre a série completa, incluindo o período de teste, e o resultado de teste da rodada 1 motivou esta análise. Por isso, os números de teste da rodada 2 devem ser lidos como levemente otimistas. Mantemos os resultados das duas rodadas no notebook.

### Evidência no treino: os níveis não são estacionários

Antes das dobras, verificamos no treino oficial (sem validação e sem teste) se as temperaturas e umidades mudam de nível ao longo do tempo. A primeira coluna mostra a deriva da média entre o primeiro e o último mês do treino, em desvios-padrão do treino. As demais mostram o teste KPSS, cuja hipótese nula é a estacionariedade, aplicado às séries horárias em nível e às suas variações horárias. O `statsmodels` limita os valores-p do KPSS ao intervalo [0,01; 0,10]: 0,01 significa "0,01 ou menos" e 0,10 significa "0,10 ou mais".

In [46]:
train_df = df.loc[splits['train'].index]
first_month = train_df[train_df.index < train_df.index.min() + pd.Timedelta(days=30)]
last_month = train_df[train_df.index > train_df.index.max() - pd.Timedelta(days=30)]
hourly_train = train_df[train_df.index.minute == 0]

evidence = []
with warnings.catch_warnings():
    warnings.simplefilter('ignore', InterpolationWarning)
    for col in TEMP_HUM_COLS:
        evidence.append({
            'variavel': col,
            'deriva_dp': (last_month[col].mean() - first_month[col].mean()) / train_df[col].std(),
            'kpss_p_nivel': kpss(hourly_train[col], regression='c', nlags='auto', result_object=True).pvalue,
            'kpss_p_variacao_1h': kpss(hourly_train[col].diff().dropna(), regression='c', nlags='auto',
                                       result_object=True).pvalue,
        })
evidence = pd.DataFrame(evidence).set_index('variavel')

print('Variáveis com estacionariedade rejeitada a 5%:',
      f"níveis {(evidence['kpss_p_nivel'] < 0.05).sum()} de {len(evidence)},",
      f"variações {(evidence['kpss_p_variacao_1h'] < 0.05).sum()} de {len(evidence)}")
evidence.round(3)

Variáveis com estacionariedade rejeitada a 5%: níveis 21 de 21, variações 0 de 21


,deriva_dp,kpss_p_nivel,kpss_p_variacao_1h
variavel,,,
kitchen_temp,1.108,0.010,0.1
kitchen_hum,-0.836,0.010,0.1
living_room_temp,0.208,0.027,0.1
living_room_hum,-0.119,0.018,0.1
laundry_temp,1.461,0.010,0.1
laundry_hum,-1.555,0.010,0.1
office_temp,1.354,0.010,0.1
office_hum,-1.120,0.010,0.1
bathroom_temp,1.428,0.010,0.1


### Validação com janelas expansivas

Dividimos o período de treino e validação em três blocos de avaliação de 21 dias (o último é a validação oficial). Em cada dobra:

- o treino é tudo o que vem antes do bloco, e os alvos são criados dentro dele (as últimas 6 linhas, cujo alvo cairia no bloco, são descartadas);
- as padronizações são reajustadas no treino da dobra;
- as redes usam os últimos 7 dias do treino da dobra para a parada antecipada, de forma que o bloco fica reservado para a avaliação;
- nenhuma dobra alcança o conjunto de teste.

A tabela abaixo também mostra, para cada dobra e representação, a fração de valores de temperatura e umidade do bloco que fica fora do intervalo entre os quantis de 0,5% e 99,5% do treino da dobra. Esse diagnóstico usa apenas as entradas, sem os alvos.

In [47]:
FOLD_BLOCKS = [('2016-03-05', '2016-03-26'), ('2016-03-26', '2016-04-16'), ('2016-04-16', '2016-05-07')]
EARLY_STOPPING_DAYS = 7
FOLD_SEEDS = range(3)
TEST_START = splits['test'].index.min()


def make_fold(start, end):
    """Treino com tudo antes de `start` e bloco de avaliação [start, end), alvos criados dentro de cada parte."""
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    assert end <= TEST_START, 'as dobras não podem alcançar o teste'
    return {'train': make_targets(df[df.index < start]),
            'val': make_targets(df[(df.index >= start) & (df.index < end)])}


def early_stopping_split(data, days=EARLY_STOPPING_DAYS):
    """Separa os últimos `days` dias de um conjunto de treino para a parada antecipada."""
    stop = data[2] > data[2].max() - pd.Timedelta(days=days)
    return subset(data, ~stop), subset(data, stop)


def out_of_range_share(representation, parts):
    """Fração de valores de temperatura e umidade do bloco fora dos quantis 0,5%–99,5% do treino da dobra."""
    values = apply_representation(df[TEMP_HUM_COLS], representation)
    train_values = values.loc[parts['train'].index].dropna()
    lo, hi = train_values.quantile(0.005), train_values.quantile(0.995)
    block = values.loc[parts['val'].index]
    return ((block < lo) | (block > hi)).to_numpy().mean()


folds = {f'dobra {k}': make_fold(start, end) for k, (start, end) in enumerate(FOLD_BLOCKS, start=1)}

pd.DataFrame([
    {'dobra': name, 'treino_ate': parts['train'].index.max(), 'bloco_inicio': parts['val'].index.min(),
     'bloco_fim': parts['val'].index.max(), 'n_treino': len(parts['train']), 'n_bloco': len(parts['val']),
     **{f'fora_da_faixa_{rep}': round(out_of_range_share(rep, parts), 3) for rep in REPRESENTATIONS}}
    for name, parts in folds.items()
]).set_index('dobra')

,treino_ate,bloco_inicio,bloco_fim,n_treino,n_bloco,fora_da_faixa_niveis,fora_da_faixa_variacoes
dobra,,,,,,,
dobra 1,2016-03-04 22:50:00,2016-03-05,2016-03-25 22:50:00,7668,3018,0.051,0.016
dobra 2,2016-03-25 22:50:00,2016-03-26,2016-04-15 22:50:00,10692,3018,0.097,0.014
dobra 3,2016-04-15 22:50:00,2016-04-16,2016-05-06 22:50:00,13716,3018,0.057,0.021


### Comparação das representações nas dobras

Em cada dobra e representação, ajustamos o ARX com a configuração da rodada 1 e treinamos o MLP com a configuração da rodada 1 e três sementes. As métricas são calculadas no bloco de avaliação da dobra; o MASE usa o mesmo denominador do restante do notebook (MAE da persistência no treino oficial), que funciona apenas como uma escala fixa por horizonte.

In [48]:
fold_started_at = perf_counter()
fold_records = []

for fold_name, parts in folds.items():
    for representation in REPRESENTATIONS:
        _, arx_fold_preds = fit_arx(arx_best_r1, representation, parts)
        fold_preds = [('arx', 0, arx_fold_preds['val'])]

        fold_sets, _ = prepare_mlp(mlp_best_L, representation, parts)
        inner_train, stopping = early_stopping_split(fold_sets['train'])
        for seed in FOLD_SEEDS:
            model, _ = train_network(inner_train, stopping, seed=seed, **mlp_best_settings)
            fold_preds.append(('mlp', seed, nn_predictions(model, {'val': fold_sets['val']})['val']))

        for model_name, seed, by_horizon in fold_preds:
            for horizon, pred in by_horizon.items():
                assert pred.index.equals(arx_fold_preds['val'][horizon].index), 'linhas diferentes entre modelos'
                fold_records.append({'dobra': fold_name, 'representacao': representation, 'modelo': model_name,
                                     'seed': seed, 'horizonte_min': 10 * HORIZONS[horizon],
                                     **regression_metrics(parts['val'].loc[pred.index, horizon], pred, horizon)})
    print(f'{fold_name} concluída ({perf_counter() - fold_started_at:.0f} s)', flush=True)

fold_results = pd.DataFrame(fold_records)
fold_by_model = (fold_results.groupby(['representacao', 'modelo', 'horizonte_min', 'dobra'])[['MAE', 'MASE']].mean()
                 .unstack('dobra'))
fold_by_model.round(3)

dobra 1 concluída (49 s)
dobra 2 concluída (112 s)
dobra 3 concluída (182 s)


MAE                     MASE          \
dobra                              dobra 1  dobra 2 dobra 3 dobra 1 dobra 2   
representacao modelo horizonte_min                                            
niveis        arx    10             32.590   27.412  27.446   1.057   0.889   
                     60             47.463   42.628  42.939   0.787   0.707   
              mlp    10             44.389   32.635  28.088   1.440   1.058   
                     60             60.045   45.858  39.621   0.996   0.761   
variacoes     arx    10             33.823   28.133  27.744   1.097   0.912   
                     60             50.362   44.355  44.870   0.835   0.736   
              mlp    10             40.331   71.568  29.429   1.308   2.321   
                     60             52.705  150.114  40.424   0.874   2.490   

                                            
dobra                              dobra 3  
representacao modelo horizonte_min          
niveis        arx    10              0.890  
                     60              0.712  
              mlp    10              0.911  
                     60              0.657  
variacoes     arx    10              0.900  
                     60              0.744  
              mlp    10              0.954  
                     60              0.671

In [49]:
fold_mase = (fold_results.groupby(['representacao', 'modelo', 'horizonte_min', 'dobra'], as_index=False)['MASE'].mean()
             .assign(modelo=lambda d: d['modelo'].map(MODEL_LABELS),
                     horizonte=lambda d: d['horizonte_min'].astype(str) + ' min'))

fig = px.line(fold_mase, x='dobra', y='MASE', color='representacao', facet_col='horizonte', facet_row='modelo',
              markers=True, template='plotly_white', title='MASE nos blocos das dobras por representação',
              labels={'representacao': 'Representação', 'dobra': ''})
clean_facet_titles(fig)
fig.update_yaxes(matches=None, showticklabels=True)
fig.show()

decision = fold_mase.groupby('representacao')['MASE'].mean()
CHOSEN_REPRESENTATION = decision.idxmin()
print('MASE médio nas dobras:', decision.round(4).to_dict())
print('Representação escolhida:', CHOSEN_REPRESENTATION)

MASE médio nas dobras: {'niveis': 0.9055, 'variacoes': 1.1536}
Representação escolhida: niveis


__Resultado.__ A regra escolheu `niveis` (MASE médio de 0,906, contra 1,174 de `variacoes`). Com `variacoes`, o ARX ficou ligeiramente pior em todas as dobras (por exemplo, MAE de 44,9 contra 42,9 Wh aos 60 minutos na dobra 3). O MLP melhorou na dobra 1 (MAE de 40,3 contra 44,4 Wh aos 10 minutos, média das sementes), mas ficou instável na dobra 2, com MAE de 60 a 94 Wh aos 10 minutos e de 132 a 201 Wh aos 60 minutos, conforme a semente. Como a regra foi definida antes de ver os resultados, mantemos `niveis`.

Duas observações ajudam a interpretar esse resultado negativo, sem alterar a decisão:

- A mudança de distribuição dentro do período de treino e validação é bem menor do que no teste: nos blocos das dobras, apenas 5% a 10% dos valores em nível ficam fora da faixa do treino da dobra. As dobras, portanto, premiam pouco uma representação robusta à mudança de distribuição.
- A instabilidade do MLP com `variacoes` na dobra 2 sugere que as variações também têm valores extremos aos quais a rede reage mal; uma hipótese são saltos bruscos de umidade.

A mudança de distribuição continua sendo o principal modo de falha documentado das redes. A seção seguinte ataca o problema por outro caminho: a cabeça residual faz a rede partir da persistência, o que limita o efeito de extrapolações.

### Consequência para os modelos seguintes

Os modelos das próximas seções usam `CHOSEN_REPRESENTATION`. Quando a representação escolhida é `niveis`, a da rodada 1, o ARX e as redes da rodada 1 já são os modelos finais nessa representação e são reaproveitados; caso contrário, o ARX é reajustado (grade e seleção na validação) e as redes usam os hiperparâmetros da rodada 1 com a nova representação.

In [50]:
if CHOSEN_REPRESENTATION == 'niveis':
    arx_best_r2, arx_preds_r2 = arx_best_r1, arx_preds_r1
else:
    arx_best_r2 = best_arx(arx_grid(CHOSEN_REPRESENTATION))
    _, arx_preds_r2 = fit_arx(arx_best_r2, CHOSEN_REPRESENTATION)
PREDICTIONS[('arx', 2)] = long_predictions('arx', 2, arx_preds_r2)
export_predictions('arx', 2)
arx_best_r2

,p,q,lag24,alpha,MAE_val
horizonte,,,,,
y_10,12,0,True,100.0,27.445966
y_60,12,0,True,1000.0,42.939072


## Rodada 2: cabeça residual, defasagem de 24 h e ARX + MLP

Uma auditoria da comparação apontou três diferenças entre as redes e o ARX que podiam explicar parte do resultado da rodada 1:

1. __Defasagem de 24 h.__ O ARX escolheu na validação a defasagem de 24 h ($y_{t+h-144}$), que o MLP e a CNN não recebiam. A opção `lag24=True` dá às redes o consumo 24 h antes de cada um dos seis instantes previstos.
2. __Cabeça residual.__ Com `residual=True`, a rede prevê $\log(1+y_{t+k}) - \log(1+y_t)$ e a cabeça soma $\log(1+y_t)$ de volta. A rede parte da persistência em vez de prever o nível do consumo, o que também limita o estrago de extrapolações fora da faixa de treino.
3. __Curvas de aprendizado.__ As perdas por época agora são registradas e plotadas para todas as redes.

Além disso, testamos um __MLP ajustado aos resíduos do ARX__: o ARX faz a previsão e o MLP tenta corrigir o erro dele.

__Regra de decisão, definida antes de ver os resultados.__ Para cada família (MLP e CNN 1D), comparamos as quatro variantes (base, residual, 24 h, residual + 24 h) nas mesmas três dobras da validação temporal, com três sementes, e escolhemos a de menor MASE médio (dobras, horizontes e sementes). Os hiperparâmetros são os da rodada 1. A variante escolhida é a que entra na comparação final; as demais aparecem só na tabela completa, para referência. Para o ARX + MLP, a correção é considerada útil se reduzir o MASE médio do ARX nas dobras.

In [51]:
NETWORK_VARIANTS = {'': {}, '_res': {'residual': True}, '_lag24': {'lag24': True},
                    '_res_lag24': {'residual': True, 'lag24': True}}
FAMILIES = {
    'mlp': {
        'prepare': lambda parts, variant: prepare_mlp(mlp_best_L, CHOSEN_REPRESENTATION, parts, **variant),
        'train': lambda variant: {**mlp_best_settings, 'residual': variant.get('residual', False)},
    },
    'cnn_1d': {
        'prepare': lambda parts, variant: prepare_cnn(cnn_best_L, CHOSEN_REPRESENTATION, parts, **variant),
        'train': lambda variant: {**cnn_best_settings, 'model_factory': build_cnn_model, **variant},
    },
}

variants_started_at = perf_counter()
variant_records = []
for fold_name, parts in folds.items():
    for family, spec in FAMILIES.items():
        reference_rows = None
        for suffix, variant in NETWORK_VARIANTS.items():
            fold_sets, _ = spec['prepare'](parts, variant)
            reference_rows = fold_sets['val'][2] if reference_rows is None else reference_rows
            assert fold_sets['val'][2].equals(reference_rows), 'variantes avaliadas em linhas diferentes'
            inner_train, stopping = early_stopping_split(fold_sets['train'])
            for seed in FOLD_SEEDS:
                model, _ = train_network(inner_train, stopping, seed=seed, **spec['train'](variant))
                for horizon, pred in nn_predictions(model, {'val': fold_sets['val']})['val'].items():
                    variant_records.append({'dobra': fold_name, 'familia': family, 'variante': family + suffix,
                                            'seed': seed, 'horizonte_min': 10 * HORIZONS[horizon],
                                            **regression_metrics(parts['val'].loc[pred.index, horizon], pred, horizon)})
    print(f'{fold_name} concluída ({perf_counter() - variants_started_at:.0f} s)', flush=True)

variant_results = pd.DataFrame(variant_records)
variant_mase = (variant_results.groupby(['familia', 'variante', 'dobra', 'horizonte_min'])['MASE'].mean()
                .groupby(['familia', 'variante']).mean())
CHOSEN_VARIANTS = {family: variant_mase[family].idxmin() for family in FAMILIES}
print('Variantes escolhidas:', CHOSEN_VARIANTS)

(variant_results.groupby(['variante', 'horizonte_min', 'dobra'])['MASE'].mean().unstack('dobra')
 .assign(media_dobras=lambda d: d.mean(axis=1)).round(3))

dobra 1 concluída (130 s)
dobra 2 concluída (280 s)
dobra 3 concluída (496 s)
Variantes escolhidas: {'mlp': 'mlp_res', 'cnn_1d': 'cnn_1d_res'}


dobra                           dobra 1  dobra 2  dobra 3  media_dobras
variante         horizonte_min                                         
cnn_1d           10               2.603    1.630    1.349         1.860
                 60               1.423    0.895    0.825         1.048
cnn_1d_lag24     10               2.398    1.632    1.285         1.772
                 60               1.366    0.891    0.775         1.011
cnn_1d_res       10               0.981    0.836    0.817         0.878
                 60               0.783    0.673    0.660         0.705
cnn_1d_res_lag24 10               0.981    0.848    0.816         0.882
                 60               0.795    0.669    0.649         0.704
mlp              10               1.440    1.058    0.911         1.136
                 60               0.996    0.761    0.657         0.805
mlp_lag24        10               1.472    1.104    0.943         1.173
                 60               1.005    0.740    0.644         0.796
mlp_res          10               0.952    0.837    0.801         0.863
                 60               0.750    0.654    0.628         0.677
mlp_res_lag24    10               0.955    0.843    0.795         0.864
                 60               0.757    0.666    0.617         0.680

In [52]:
fig = px.line(variant_results.groupby(['familia', 'variante', 'dobra', 'horizonte_min'], as_index=False)['MASE'].mean()
              .assign(variante=lambda d: d['variante'].map(MODEL_LABELS),
                      horizonte=lambda d: d['horizonte_min'].astype(str) + ' min'),
              x='dobra', y='MASE', color='variante', facet_col='horizonte', facet_row='familia', markers=True,
              template='plotly_white', title='MASE nos blocos das dobras por variante', labels={'dobra': ''})
clean_facet_titles(fig)
fig.update_yaxes(matches=None, showticklabels=True)
fig.show()

__Resultado.__ A cabeça residual é a mudança decisiva. Nas dobras, o MASE médio do MLP caiu de 0,971 (base) para 0,770 (residual), e o da CNN 1D, de 1,454 para 0,792; a melhora aparece nas três dobras e nos dois horizontes. A defasagem de 24 h, por outro lado, praticamente não muda o resultado: 0,985 (MLP com 24 h) e 0,772 (residual + 24 h) no MLP, 1,393 e 0,794 na CNN. Pela regra, as variantes escolhidas são __MLP (residual)__ e __CNN 1D (residual)__. A falta da defasagem de 24 h, portanto, não explicava a vantagem do ARX; a falta de uma âncora na persistência, sim.

### Variantes na divisão oficial

Treinamos todas as variantes com cinco sementes na divisão oficial (parada antecipada na validação) e registramos as previsões na rodada 2. As variantes base reaproveitam os modelos da rodada 1 quando a representação não mudou. Mostramos as curvas de aprendizado das variantes escolhidas.

In [53]:
round1_networks = {'mlp': (mlp_models_r1, mlp_runs_r1, mlp_final_sets_r1),
                   'cnn_1d': (cnn_models_r1, cnn_runs_r1, cnn_final_sets_r1)}
variant_models, variant_runs = {}, {}
for family, spec in FAMILIES.items():
    for suffix, variant in NETWORK_VARIANTS.items():
        key = family + suffix
        print(f'--- {MODEL_LABELS[key]}', flush=True)
        if suffix == '' and CHOSEN_REPRESENTATION == 'niveis':
            models, runs, sets = round1_networks[family]
        else:
            sets, scaler = spec['prepare'](splits, variant)
            models, runs = train_seeds(sets, **spec['train'](variant))
            export_scaler(scaler, key, 2)
        variant_models[key], variant_runs[key] = models, runs
        PREDICTIONS[(key, 2)] = seed_predictions(key, 2, models, sets)
        export_predictions(key, 2)

for key in CHOSEN_VARIANTS.values():
    plot_learning_curves(variant_models[key], f'Curvas de aprendizado: {MODEL_LABELS[key]} (rodada 2)')

--- MLP
--- MLP (residual)
Semente 0: treino concluído em 7.0 s
Semente 1: treino concluído em 7.1 s
Semente 2: treino concluído em 8.7 s
Semente 3: treino concluído em 5.4 s
Semente 4: treino concluído em 7.0 s
--- MLP (24 h)
Semente 0: treino concluído em 12.6 s
Semente 1: treino concluído em 13.0 s
Semente 2: treino concluído em 15.7 s
Semente 3: treino concluído em 12.7 s
Semente 4: treino concluído em 19.3 s
--- MLP (residual + 24 h)
Semente 0: treino concluído em 7.1 s
Semente 1: treino concluído em 6.7 s
Semente 2: treino concluído em 6.2 s
Semente 3: treino concluído em 8.1 s
Semente 4: treino concluído em 7.4 s
--- CNN 1D
--- CNN 1D (residual)
Semente 0: treino concluído em 7.7 s
Semente 1: treino concluído em 8.1 s
Semente 2: treino concluído em 7.7 s
Semente 3: treino concluído em 8.6 s
Semente 4: treino concluído em 7.2 s
--- CNN 1D (24 h)
Semente 0: treino concluído em 23.2 s
Semente 1: treino concluído em 15.9 s
Semente 2: treino concluído em 13.2 s
Semente 3: treino conc

### MLP ajustado aos resíduos do ARX

O ARX faz a previsão de cada horizonte e um MLP prevê o erro dele, $e_{t+h} = y_{t+h} - \hat{y}^{\text{ARX}}_{t+h}$; a previsão final é $\hat{y}^{\text{ARX}}_{t+h} + \hat{e}_{t+h}$. O ponto delicado são os resíduos do treino: os resíduos do ARX ajustado no próprio treino são menores que os erros reais de previsão, e um MLP treinado neles aprenderia a corrigir pouco. Por isso, os resíduos de treino são __fora da amostra__: dividimos o treino em blocos mensais e cada bloco é previsto por um ARX ajustado só nos dados anteriores a ele (o primeiro mês, sem previsão, fica de fora). Validação e teste usam o ARX ajustado no treino inteiro.

O MLP recebe as mesmas entradas do MLP da rodada 1 e as duas previsões do ARX, e prevê os resíduos dos dois horizontes (divididos pelo desvio-padrão dos resíduos do treino) com perda L1, coerente com o MAE. A comparação com "prever zero" (o ARX sozinho) é feita nas dobras e depois na divisão oficial.

In [54]:
def arx_oos_predictions(best, representation, parts):
    """Previsões do ARX: fora da amostra no treino (blocos mensais expansivos) e do ajuste completo nos demais."""
    train = parts['train']
    starts = pd.date_range(train.index.min(), train.index.max(), freq='MS')
    ends = list(starts[1:]) + [train.index.max() + pd.Timedelta(minutes=10)]
    blocks = {h: [] for h in TARGETS}
    for start, end in zip(starts, ends):
        block = {'train': make_targets(df[df.index < start]), 'val': train[(train.index >= start) & (train.index < end)]}
        _, preds = fit_arx(best, representation, block)
        for h in TARGETS:
            blocks[h].append(preds['val'][h])
    _, full = fit_arx(best, representation, parts)
    oos = {'train': pd.DataFrame({h: pd.concat(blocks[h]) for h in TARGETS})}
    return {**oos, **{name: pd.DataFrame(full[name]) for name in parts if name != 'train'}}


def prepare_arx_residuals(parts, best=arx_best_r2, representation=CHOSEN_REPRESENTATION):
    """Entradas do MLP + previsões do ARX, resíduos padronizados como alvo, e as previsões do ARX por conjunto."""
    arx = arx_oos_predictions(best, representation, parts)
    base, _ = prepare_mlp(mlp_best_L, representation, parts)
    rows = {name: base[name][2].intersection(arx[name].dropna().index) for name in parts}
    arx_scaler = StandardScaler().fit(arx['train'].loc[rows['train']])
    residuals = {name: parts[name].loc[rows[name], TARGETS] - arx[name].loc[rows[name]] for name in parts}
    scale = residuals['train'].std().to_numpy(dtype=np.float32)

    sets = {}
    for name, (inputs, _, index) in base.items():
        mask = index.isin(rows[name])
        extra = arx_scaler.transform(arx[name].loc[rows[name]]).astype(np.float32)
        sets[name] = (np.hstack([inputs[mask], extra]),
                      (residuals[name].to_numpy(dtype=np.float32) / scale).astype(np.float32), rows[name])
    return sets, {name: arx[name].loc[rows[name]] for name in parts}, scale


def arx_mlp_predictions(model, sets, arx_preds, scale):
    """Previsão do ARX mais a correção do MLP, em Wh: {conjunto: {horizonte: Series}}."""
    preds = {}
    for name, (inputs, _, index) in sets.items():
        correction = predict_raw(model, inputs) * scale
        preds[name] = {h: arx_preds[name][h] + correction[:, j] for j, h in enumerate(TARGETS)}
    return preds


arx_mlp_records = []
for fold_name, parts in folds.items():
    fold_sets, fold_arx, fold_scale = prepare_arx_residuals(parts)
    inner_train, stopping = early_stopping_split(fold_sets['train'])
    candidates = [('arx', 0, {h: fold_arx['val'][h] for h in TARGETS})]
    for seed in FOLD_SEEDS:
        model, _ = train_network(inner_train, stopping, seed=seed, loss='l1', **mlp_best_settings)
        candidates.append(('arx_mlp', seed, arx_mlp_predictions(model, {'val': fold_sets['val']}, fold_arx, fold_scale)['val']))
    for model_name, seed, by_horizon in candidates:
        for horizon, pred in by_horizon.items():
            arx_mlp_records.append({'dobra': fold_name, 'modelo': model_name, 'seed': seed,
                                    'horizonte_min': 10 * HORIZONS[horizon],
                                    **regression_metrics(parts['val'].loc[pred.index, horizon], pred, horizon)})

arx_mlp_folds = pd.DataFrame(arx_mlp_records)
arx_mlp_mase = (arx_mlp_folds.groupby(['modelo', 'dobra', 'horizonte_min'])['MASE'].mean().groupby('modelo').mean())
print('MASE médio nas dobras:', arx_mlp_mase.round(4).to_dict())
print('A correção do MLP reduz o MASE do ARX nas dobras:', bool(arx_mlp_mase['arx_mlp'] < arx_mlp_mase['arx']))
arx_mlp_folds.groupby(['modelo', 'horizonte_min', 'dobra'])['MASE'].mean().unstack('dobra').round(3)

MASE médio nas dobras: {'arx': 0.8405, 'arx_mlp': 0.8305}
A correção do MLP reduz o MASE do ARX nas dobras: True


dobra                  dobra 1  dobra 2  dobra 3
modelo  horizonte_min                           
arx     10               1.057    0.889    0.890
        60               0.787    0.707    0.712
arx_mlp 10               1.011    0.875    0.870
        60               0.939    0.634    0.654

In [55]:
arx_mlp_sets, arx_mlp_arx, arx_mlp_scale = prepare_arx_residuals(splits)
arx_mlp_models, arx_mlp_runs = train_seeds(arx_mlp_sets, loss='l1', **mlp_best_settings)
PREDICTIONS[('arx_mlp', 2)] = pd.concat(
    [long_predictions('arx_mlp', 2, arx_mlp_predictions(model, arx_mlp_sets, arx_mlp_arx, arx_mlp_scale), seed=seed)
     for seed, model in arx_mlp_models.items()], ignore_index=True)
export_predictions('arx_mlp', 2)

plot_learning_curves(arx_mlp_models, 'Curvas de aprendizado do MLP sobre os resíduos do ARX')
summarize_seeds(metrics_by_seed(predictions_table(rodada=2).query("modelo in ['arx', 'arx_mlp']"))).round(3)

Semente 0: treino concluído em 9.1 s
Semente 1: treino concluído em 8.1 s
Semente 2: treino concluído em 5.5 s
Semente 3: treino concluído em 7.0 s
Semente 4: treino concluído em 7.8 s


MAE           RMSE          MASE  \
                                       média     DP   média     DP  média   
rodada modelo    split horizonte_min                                        
2      ARX       train 10             31.646    NaN  66.776    NaN  1.026   
                       60             50.129    NaN  92.532    NaN  0.832   
                 val   10             27.446    NaN  60.583    NaN  0.890   
                       60             42.939    NaN  84.354    NaN  0.712   
                 test  10             27.083    NaN  59.440    NaN  0.878   
                       60             40.270    NaN  78.386    NaN  0.668   
       ARX + MLP train 10             25.906  0.442  64.507  0.663  0.840   
                       60             35.160  1.264  83.943  2.006  0.583   
                 val   10             25.237  0.205  60.548  0.202  0.818   
                       60             38.097  0.313  86.944  0.922  0.632   
                 test  10             24.552  0.396  59.290  0.341  0.796   
                       60             35.549  0.754  81.705  1.185  0.590   

                                             
                                         DP  
rodada modelo    split horizonte_min         
2      ARX       train 10               NaN  
                       60               NaN  
                 val   10               NaN  
                       60               NaN  
                 test  10               NaN  
                       60               NaN  
       ARX + MLP train 10             0.014  
                       60             0.021  
                 val   10             0.007  
                       60             0.005  
                 test  10             0.013  
                       60             0.013

__Resultado.__ Nas dobras, a correção do MLP reduziu o MASE médio do ARX de 0,841 para 0,831, uma melhora pequena e desigual: aos 10 minutos ela aparece nas três dobras (por exemplo, MAE de 26,8 contra 27,5 Wh na dobra 3), mas aos 60 minutos a correção piorou bastante a dobra 1 (56,6 contra 47,5 Wh), a de treino mais curto, e melhorou as dobras 2 e 3 (38,2 contra 42,6 e 39,4 contra 42,9 Wh). Pela regra definida antes, a correção é considerada útil. Na divisão oficial, o ARX + MLP tem MAE de teste de 24,6 ± 0,4 e 35,6 ± 0,8 Wh, contra 27,1 e 40,3 Wh do ARX sozinho, ou seja, o MLP encontra estrutura nos resíduos do ARX que o modelo linear não captura.

## Modelo principal: rede recorrente (GRU)

__Entradas.__ Janelas no formato `(amostras, L, variáveis)` (`batch_first=True`) com, em cada passo: $\log(1+y)$, as luzes, as temperaturas e umidades (na representação escolhida), as variáveis meteorológicas da última hora cheia e o seno e o cosseno da hora do dia e do dia da semana. As variáveis são padronizadas com o treino. Além da janela, a rede recebe um vetor com as variáveis de calendário dos instantes previstos ($t+1$ e $t+6$), conhecidas de antemão, e $\log(1+y_t)$ para a cabeça residual. As origens de validação e teste são as mesmas de todos os outros modelos.

__Arquitetura.__ GRU unidirecional (uma rede bidirecional usaria o futuro) e sem estado entre janelas (estado inicial nulo em cada janela), com 1 ou 2 camadas. O último estado oculto é concatenado ao vetor de calendário dos instantes previstos e passa por uma camada linear. A cabeça residual soma $\log(1+y_t)$ à saída, de forma que a rede parte da persistência. Dropout entre as camadas recorrentes e antes da camada linear.

__Treinamento.__ Perda L1 sobre $\log(1+y)$: como $\exp(\cdot) - 1$ é monótona, a mediana prevista na escala log é a mediana em Wh, que é o que minimiza o MAE reportado. AdamW (taxa `0,001`, decaimento de pesos `1e-4`), corte da norma do gradiente em 1,0 e parada antecipada na validação, que vem depois do treino.

__Busca.__ Janela $L \in \{36, 72, 144\}$ (6 a 24 h), tamanho oculto em $\{32, 64, 128\}$, 1 ou 2 camadas e dropout em $\{0{,}1; 0{,}2\}$, com a semente `0` e o MAE médio de validação (10 e 60 min) como critério. Na melhor configuração, comparamos um modelo compartilhado (saídas para $t+1$ e $t+6$) com um modelo separado por horizonte. A configuração final é treinada com cinco sementes, e a mesma configuração com `nn.LSTM` serve de ablação.

__Execução.__ A busca completa é pesada para CPU. A grade é definida pelas constantes `GRU_*` da célula abaixo, e a GRU usa GPU quando disponível (`DEVICE`). No Colab, use um ambiente com GPU e a célula de configuração do início do notebook.

In [56]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

GRU_LOOKBACKS = (36, 72, 144)
GRU_HIDDEN = (32, 64, 128)
GRU_LAYERS = (1, 2)
GRU_DROPOUT = (0.1, 0.2)
GRU_OUTPUT_STEPS = (1, 6)
GRU_TRAIN = {'loss': 'l1', 'grad_clip': 1.0, 'lr': 1e-3, 'max_epochs': 200, 'device': DEVICE,
             'model_factory': None}  # model_factory é definido junto com a arquitetura

print('Dispositivo da GRU:', DEVICE)

Dispositivo da GRU: cuda


#### Janelas da GRU

In [57]:
TIME_COLS = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']


def time_features(index):
    """Seno e cosseno da hora do dia e do dia da semana."""
    minutes = index.hour * 60 + index.minute
    day = index.day_of_week + minutes / 1440
    return pd.DataFrame({'hour_sin': np.sin(2 * np.pi * minutes / 1440), 'hour_cos': np.cos(2 * np.pi * minutes / 1440),
                         'dow_sin': np.sin(2 * np.pi * day / 7), 'dow_cos': np.cos(2 * np.pi * day / 7)}, index=index)


def prepare_gru(L, representation=CHOSEN_REPRESENTATION, parts=splits, output_steps=GRU_OUTPUT_STEPS):
    """Entradas (janelas (n, L, F), calendário dos instantes previstos, log1p(y_t)), alvos log1p e origens."""
    measured = df.drop(columns=[c for c in df.columns if c.startswith(CALENDAR_PREFIXES)])
    frame = apply_representation(measured, representation)
    frame = frame.assign(appliances=np.log1p(frame['appliances'])).join(time_features(df.index))

    base, _ = prepare_mlp(MLP_HISTORY, representation, parts)  # mesmas origens e alvos das outras redes
    scaler = StandardScaler().fit(frame.loc[base['train'][2]])
    windows = sliding_window_view(scaler.transform(frame).astype(np.float32), window_shape=L, axis=0)
    window_end_index = frame.index[L - 1:]
    step = pd.Timedelta(minutes=10)

    sets = {}
    for name, (_, targets, origins) in base.items():
        positions = window_end_index.get_indexer(origins)
        valid = positions >= 0
        x = np.zeros((len(origins), L, frame.shape[1]), dtype=np.float32)
        x[valid] = windows[positions[valid]].transpose(0, 2, 1)
        vector = np.hstack([time_features(origins + k * step).to_numpy(dtype=np.float32) for k in output_steps])
        current = np.log1p(df.loc[origins, 'appliances'].to_numpy(dtype=np.float32))[:, None]
        complete = valid & np.isfinite(x).all(axis=(1, 2))
        if name != 'train' and not complete.all():
            raise ValueError(f'Janelas incompletas em {name} para L={L}.')
        data = ((x, vector, current), targets[:, [k - 1 for k in output_steps]].copy(), origins)
        sets[name] = subset(data, complete)
    return sets, scaler


gru_check_sets, _ = prepare_gru(GRU_LOOKBACKS[0])
for name in ('val', 'test'):
    pd.testing.assert_index_equal(gru_check_sets[name][2], mlp_final_sets_r1[name][2])
pd.DataFrame.from_dict({name: {'janelas': inputs[0].shape, 'vetor': inputs[1].shape, 'y': targets.shape}
                        for name, (inputs, targets, _) in gru_check_sets.items()}, orient='index')

,janelas,vetor,y
train,"(13681, 36, 30)","(13681, 8)","(13681, 2)"
val,"(3018, 36, 30)","(3018, 8)","(3018, 2)"
test,"(2983, 36, 30)","(2983, 8)","(2983, 2)"


#### Arquitetura da GRU

In [58]:
class RecurrentForecaster(nn.Module):
    def __init__(self, n_features, n_vector, n_out, hidden=64, layers=1, dropout=0.1, cell='gru'):
        super().__init__()
        recurrent = {'gru': nn.GRU, 'lstm': nn.LSTM}[cell]
        self.rnn = recurrent(n_features, hidden, num_layers=layers, batch_first=True,
                             dropout=dropout if layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden + n_vector, n_out)

    def forward(self, x, vector):
        output, _ = self.rnn(x)  # estado inicial nulo em cada janela
        return self.head(self.dropout(torch.cat([output[:, -1], vector], dim=1)))


def build_recurrent_model(inputs, n_out, hidden=64, layers=1, dropout=0.1, cell='gru', **_):
    windows, vector, _ = inputs
    model = RecurrentForecaster(windows.shape[2], vector.shape[1], n_out, hidden, layers, dropout, cell)
    assert not model.rnn.bidirectional
    return ResidualOutput(model)


GRU_TRAIN['model_factory'] = build_recurrent_model

gru_check = build_recurrent_model(gru_check_sets['train'][0], len(GRU_OUTPUT_STEPS)).eval()
with torch.no_grad():
    gru_example = gru_check(*[torch.from_numpy(x[:4]) for x in gru_check_sets['train'][0]])
print('Dimensões da saída:', tuple(gru_example.shape))
print('Número de parâmetros:', sum(p.numel() for p in gru_check.parameters()))

Dimensões da saída: (4, 2)
Número de parâmetros: 18578


#### Busca de hiperparâmetros da GRU

In [59]:
GRU_SETTINGS_GRID = [{'hidden': hidden, 'layers': layers, 'dropout': dropout}
                     for hidden, layers, dropout in product(GRU_HIDDEN, GRU_LAYERS, GRU_DROPOUT)]

gru_grid = network_grid(lambda L: prepare_gru(L)[0], GRU_LOOKBACKS, GRU_SETTINGS_GRID,
                        output_steps=GRU_OUTPUT_STEPS, **GRU_TRAIN)
gru_L, gru_settings = best_config(gru_grid, ['hidden', 'layers', 'dropout'])
gru_grid.head(10).round(4)

L = 36: 12 configurações concluídas.
L = 72: 12 configurações concluídas.
L = 144: 12 configurações concluídas.
Tempo total da grade: 368.8 s


,L,hidden,layers,dropout,mae_10_wh,mae_60_wh,mae_medio_wh,melhor_epoca,tempo_treino_s
0,72,128,2,0.1,23.7136,35.8996,29.8066,9,9.4228
1,36,128,2,0.2,24.2507,35.8025,30.0266,8,8.4637
2,36,32,2,0.2,24.2592,35.9113,30.0852,18,11.7622
3,72,128,2,0.2,24.1168,36.1477,30.1323,9,9.4280
4,36,128,2,0.1,24.0220,36.2945,30.1582,8,8.4649
5,36,32,2,0.1,24.3753,36.0313,30.2033,12,9.7197
6,72,32,2,0.2,24.3073,36.1746,30.2409,15,11.5308
7,144,32,2,0.1,24.3330,36.2015,30.2672,15,13.4047
8,36,128,1,0.2,24.4815,36.0834,30.2824,9,8.1166
9,36,128,1,0.1,24.4508,36.1184,30.2846,7,7.4858


#### Modelo compartilhado ou um modelo por horizonte

Na melhor configuração da grade, comparamos na validação (semente `0`) o modelo compartilhado, com saídas para $t+1$ e $t+6$, com dois modelos separados, cada um com uma saída e com o calendário do seu instante previsto.

In [60]:
def gru_sets_for(output_steps):
    return prepare_gru(gru_L, output_steps=output_steps)[0]


def val_mae(model, sets, output_steps):
    preds = nn_predictions(model, {'val': sets['val']}, output_steps)['val']
    return {h: regression_metrics(splits['val'].loc[p.index, h], p, h)['MAE'] for h, p in preds.items()}


gru_sets = gru_sets_for(GRU_OUTPUT_STEPS)
shared_model, _ = train_network(gru_sets['train'], gru_sets['val'], seed=0, **GRU_TRAIN, **gru_settings)
gru_form_mae = {'compartilhado': val_mae(shared_model, gru_sets, GRU_OUTPUT_STEPS), 'separado': {}}
for horizon, steps in HORIZONS.items():
    sets_h = gru_sets_for((steps,))
    model_h, _ = train_network(sets_h['train'], sets_h['val'], seed=0, **GRU_TRAIN, **gru_settings)
    gru_form_mae['separado'].update(val_mae(model_h, sets_h, (steps,)))

gru_form = pd.DataFrame(gru_form_mae).T.assign(media=lambda d: d.mean(axis=1))
GRU_SHARED_MODEL = gru_form.loc['compartilhado', 'media'] <= gru_form.loc['separado', 'media']
print('Forma escolhida:', 'compartilhado' if GRU_SHARED_MODEL else 'separado por horizonte')
gru_form.round(3)

Forma escolhida: compartilhado


,y_10,y_60,media
compartilhado,23.714,35.900,29.807
separado,23.284,36.379,29.831


#### Treinamento final da GRU e ablação com LSTM

Treinamos a configuração escolhida com as sementes de 0 a 4, e a mesma configuração com `nn.LSTM` no lugar de `nn.GRU`.

In [61]:
def train_recurrent(cell):
    """Treina a configuração escolhida (compartilhada ou por horizonte) com cinco sementes."""
    settings = {**GRU_TRAIN, **gru_settings, 'cell': cell}
    forms = [GRU_OUTPUT_STEPS] if GRU_SHARED_MODEL else [(steps,) for steps in HORIZONS.values()]
    models, runs, tables = {}, [], []
    for output_steps in forms:
        sets = gru_sets_for(output_steps)
        models[output_steps], runs_form = train_seeds(sets, **settings)
        runs.append(runs_form.assign(saidas=str(output_steps)))
        tables.append(seed_predictions(cell, 2, models[output_steps], sets, output_steps))
    PREDICTIONS[(cell, 2)] = pd.concat(tables, ignore_index=True)
    export_predictions(cell, 2)
    return models, pd.concat(runs)


gru_models, gru_runs = train_recurrent('gru')
lstm_models, lstm_runs = train_recurrent('lstm')

for output_steps, models in gru_models.items():
    plot_learning_curves(models, f'Curvas de aprendizado da GRU (saídas {output_steps})')
summarize_seeds(metrics_by_seed(predictions_table(rodada=2).query("modelo in ['gru', 'lstm']"))).round(3)

Semente 0: treino concluído em 9.3 s
Semente 1: treino concluído em 8.6 s
Semente 2: treino concluído em 8.2 s
Semente 3: treino concluído em 8.9 s
Semente 4: treino concluído em 9.4 s
Semente 0: treino concluído em 8.5 s
Semente 1: treino concluído em 8.1 s
Semente 2: treino concluído em 8.5 s
Semente 3: treino concluído em 11.2 s
Semente 4: treino concluído em 9.9 s


MAE           RMSE          MASE       
                                    média     DP   média     DP  média     DP
rodada modelo split horizonte_min                                            
2      GRU    train 10             27.617  0.269  67.697  0.726  0.896  0.009
                    60             40.383  0.927  95.483  2.541  0.670  0.015
              val   10             24.180  0.299  61.991  0.773  0.784  0.010
                    60             36.307  0.276  87.326  1.876  0.602  0.005
              test  10             23.592  0.299  60.482  0.478  0.765  0.010
                    60             34.878  0.639  81.554  1.597  0.579  0.011
       LSTM   train 10             27.667  0.778  67.407  1.496  0.897  0.025
                    60             39.105  2.935  92.593  5.020  0.649  0.049
              val   10             24.598  0.119  62.389  0.239  0.798  0.004
                    60             36.972  0.219  87.834  0.498  0.613  0.004
              test  10             24.210  0.127  61.119  0.567  0.785  0.004
                    60             35.792  0.937  83.649  2.445  0.594  0.016

__Resultados da GRU.__ A busca completa da GRU e da LSTM deve ser executada em um ambiente com GPU (veja a célula de configuração do Colab no início do notebook). A interpretação desta seção será escrita a partir dessa execução. Em uma CPU, uma versão reduzida da grade serve apenas para verificar que o código funciona de ponta a ponta.

## Comparação final

A tabela completa traz todos os modelos e variantes de teste nas duas rodadas (média e desvio-padrão amostral entre sementes para as redes). Os gráficos e diagnósticos seguintes são da rodada 2, com os modelos escolhidos: persistência, ARX, ARX + MLP, as variantes escolhidas do MLP e da CNN 1D, a GRU e a LSTM.

In [62]:
final_summary = summarize_seeds(metrics_by_seed(predictions_table(split='test')))
display(final_summary.droplevel('split')[[('MAE', 'média'), ('MAE', 'DP'), ('MASE', 'média'), ('RMSE', 'média')]]
        .unstack('horizonte_min').round(3))

FINAL_MODELS = ['persistencia', 'arx', 'arx_mlp', CHOSEN_VARIANTS['mlp'], CHOSEN_VARIANTS['cnn_1d'], 'gru', 'lstm']
test_r2 = predictions_table(rodada=2, split='test').query('modelo in @FINAL_MODELS')
plot_metric_bars(test_r2, 'Comparação dos modelos no conjunto de teste (rodada 2)')

MAE                           MASE         \
                                  média              DP          média          
horizonte_min                        10      60      10      60     10     60   
rodada modelo                                                                   
1      ARX                       27.083  40.270     NaN     NaN  0.878  0.668   
       CNN 1D                    83.555  91.135  18.515  11.723  2.710  1.512   
       MLP                       36.526  52.376   3.280   5.593  1.185  0.869   
       Persistência              26.510  46.024     NaN     NaN  0.860  0.763   
2      ARX                       27.083  40.270     NaN     NaN  0.878  0.668   
       ARX + MLP                 24.552  35.549   0.396   0.754  0.796  0.590   
       CNN 1D                    83.555  91.135  18.515  11.723  2.710  1.512   
       CNN 1D (24 h)             88.296  87.024  18.916  15.796  2.864  1.444   
       CNN 1D (residual + 24 h)  25.948  39.363   0.429   1.028  0.842  0.653   
       CNN 1D (residual)         25.006  37.284   0.244   0.782  0.811  0.618   
       GRU                       23.592  34.878   0.299   0.639  0.765  0.579   
       LSTM                      24.210  35.792   0.127   0.937  0.785  0.594   
       MLP                       36.526  52.376   3.280   5.593  1.185  0.869   
       MLP (24 h)                35.856  48.907   3.860   3.100  1.163  0.811   
       MLP (residual + 24 h)     24.359  35.478   0.108   0.776  0.790  0.589   
       MLP (residual)            24.191  35.352   0.163   0.832  0.785  0.586   
       Persistência              26.510  46.024     NaN     NaN  0.860  0.763   

                                    RMSE           
                                   média           
horizonte_min                         10       60  
rodada modelo                                      
1      ARX                        59.440   78.386  
       CNN 1D                    122.131  130.203  
       MLP                        66.073   86.363  
       Persistência               66.463  101.289  
2      ARX                        59.440   78.386  
       ARX + MLP                  59.290   81.705  
       CNN 1D                    122.131  130.203  
       CNN 1D (24 h)             125.312  123.080  
       CNN 1D (residual + 24 h)   60.431   81.337  
       CNN 1D (residual)          60.937   83.133  
       GRU                        60.482   81.554  
       LSTM                       61.119   83.649  
       MLP                        66.073   86.363  
       MLP (24 h)                 64.914   83.084  
       MLP (residual + 24 h)      60.089   79.992  
       MLP (residual)             60.065   81.133  
       Persistência               66.463  101.289

In [63]:
plot_by_hour(test_r2, 'erro_abs_wh', 'MAE no teste por hora do instante previsto (rodada 2)', 'MAE (Wh)')
plot_by_hour(test_r2, 'residuo_wh', 'Resíduo médio no teste por hora do instante previsto (rodada 2)', 'Resíduo médio (Wh)')
mean_bias(test_r2).round(3)

horizonte,10 min,60 min
modelo,,
Persistência,0.013,0.365
ARX,-0.091,-1.046
ARX + MLP,5.175,14.504
MLP (residual),6.149,12.500
CNN 1D (residual),7.484,15.373
GRU,6.233,14.740
LSTM,7.636,16.544


In [64]:
display(residual_diagnostics(test_r2, 'Autocorrelação dos resíduos no teste (rodada 2)'))
plot_residuals_vs_fitted(test_r2, 'Resíduos vs. previsões no teste (rodada 2)')

,horizonte_min,modelo,n_series,lags,p_min,p_max
0,10,ARX,1,144,4.61e-05,4.61e-05
1,10,ARX + MLP,5,144,1.41e-16,1.79e-09
2,10,CNN 1D (residual),5,144,4.68e-52,1.24e-20
3,10,GRU,5,144,5.44e-25,1.97e-16
4,10,LSTM,5,144,5.90e-70,4.46e-22
5,10,MLP (residual),5,144,1.05e-27,7.32e-13
6,10,Persistência,1,144,5.20e-51,5.20e-51
7,60,ARX,1,24,8.40e-01,8.40e-01
8,60,ARX + MLP,5,24,3.23e-10,8.79e-02
9,60,CNN 1D (residual),5,24,6.84e-09,3.71e-04


In [65]:
cost_summary({'MLP (rodada 1)': mlp_runs_r1, 'CNN 1D (rodada 1)': cnn_runs_r1,
              **{f'{MODEL_LABELS[k]} (rodada 2)': variant_runs[k] for k in CHOSEN_VARIANTS.values()},
              'ARX + MLP': arx_mlp_runs, 'GRU': gru_runs, 'LSTM': lstm_runs}).round(3)

,parametros,treino_media_s,treino_dp_s,inferencia_media_ms,inferencia_dp_ms
modelo,,,,,
MLP (rodada 1),23686,16.327,2.346,0.855,0.049
CNN 1D (rodada 1),7750,17.010,3.463,2.012,0.529
MLP (residual) (rodada 2),23686,7.022,1.179,0.948,0.067
CNN 1D (residual) (rodada 2),7750,7.852,0.539,1.883,0.233
ARX + MLP,23426,7.515,1.345,0.796,0.025
GRU,160786,8.886,0.529,13.709,0.051
LSTM,214290,9.233,1.271,18.162,0.045


__Resumo.__ Com a cabeça residual, as redes passam do último para o primeiro lugar. No teste, o MLP (residual) teve MAE de 24,2 ± 0,2 Wh aos 10 minutos e 35,4 ± 0,8 Wh aos 60 minutos, contra 26,5 e 46,0 Wh da persistência e 27,1 e 40,3 Wh do ARX; a CNN 1D (residual) teve 25,0 ± 0,2 e 37,4 ± 0,6 Wh, e o ARX + MLP, 24,6 ± 0,4 e 35,6 ± 0,8 Wh. A variação entre sementes caiu de ±3,3 Wh (MLP da rodada 1, 10 minutos) para ±0,2 Wh, e o desempenho no teste ficou próximo ao da validação (24,6 e 37,6 Wh para o MLP residual), sinal de que a âncora na persistência neutralizou a maior parte do efeito da mudança de distribuição. Em RMSE, o ARX continua competitivo (59,4 e 78,4 Wh, contra 60,1 e 81,1 Wh do MLP residual), porque as redes erram mais nos picos (veja a seção seguinte).

Os valores acima vêm de uma execução completa em CPU; execuções em outro hardware podem variar levemente. Os resultados da GRU e da LSTM dependem da busca completa, que deve rodar com GPU.

## Quando o modelo de aprendizagem profunda falha?

__1. Mudança na distribuição das entradas (rodada 1).__ O MLP e a CNN 1D que preveem o nível do consumo foram bem na validação e falharam no teste: o MAE do MLP aos 60 minutos foi de 38,1 Wh na validação para 52,4 ± 5,6 Wh no teste, e o da CNN, de 43,1 para 91,4 ± 11,9 Wh. O teste (maio) é muito mais quente que o treino, e grande parte das temperaturas e umidades do teste fica fora da faixa vista no treino. Fora dessa faixa, as redes extrapolam de forma arbitrária, cada semente de um jeito, e passam a superestimar o consumo (resíduo médio de −64 Wh para a CNN). O ARX, linear e com coeficientes encolhidos, manteve o desempenho (40,3 Wh).

__2. Uma correção que não se confirmou.__ Trocar os níveis de temperatura e umidade por variações (`variacoes`) eliminaria a mudança de distribuição nas entradas, mas não melhorou os modelos na validação temporal, e mantivemos os níveis. Parte da explicação é que a mudança de distribuição dentro do período de treino e validação é bem menor que no teste, então a validação não reproduz a situação em que as redes falham. É uma limitação de qualquer validação feita só com dados do passado.

__3. O que resolveu.__ Com a cabeça residual, as redes preveem a variação em relação ao consumo atual em vez do nível, e passaram a superar a persistência e o ARX nos dois horizontes, com pouca variação entre sementes e desempenho de teste próximo ao da validação.

__4. Onde as redes ainda falham: os picos.__ O ganho das redes residuais vem das leituras comuns. No teste, nas leituras de até 300 Wh (cerca de 96% das observações), o MAE aos 60 minutos é de 22,4 Wh para o MLP residual, contra 28,6 Wh do ARX e 34,7 Wh da persistência. Nos picos acima de 300 Wh (4%), o MLP residual erra mais que os dois (MAE de 320 Wh, contra 296 Wh do ARX e da persistência). Os piores horários são o fim da tarde, às 17h e às 18h, o pico de consumo do dia. A perda L1/Huber sobre $\log(1+y)$ aproxima a mediana condicional e subestima sistematicamente esses eventos (resíduo médio positivo, de +6 e +12 Wh), e os resíduos de 10 minutos ainda têm autocorrelação (teste de Ljung–Box). Prever os picos, eventos raros e de início abrupto, continua sendo o principal limite dos modelos.

In [3]:
!git config --global user.email "ian.barreiro@gmail.com"
!git config --global user.name "iantextbar"

In [5]:
# Stage all changes
!git add .

# Commit with a message
!git commit -m "GRU trained"

# Push to the remote repository
!git push origin refactor/validacao-temporal

On branch refactor/validacao-temporal
Your branch is up to date with 'origin/refactor/validacao-temporal'.

nothing to commit, working tree clean
fatal: could not read Username for 'https://github.com': No such device or address
